# Text2Cypher Toolkit: Final Evaluation on Kaggle (T4 x2)

Kaggle counterpart of `final-evaluation.ipynb`. Same protocol and seeded rows. Jobs are reused from the Hub only when they ran on the same commit and platform (see the re-run plan below), so the bundle is a single-protocol result.

### How the two GPUs are used
- **GPU workers (one per T4):** pull jobs from a shared queue and run `app.py predict` with `CUDA_VISIBLE_DEVICES` set to their own GPU. Each process sees exactly one device.
- **Evaluation workers (CPU / network):** run `app.py evaluate --execute --validate-syntax` as soon as a prediction file is ready, so a GPU never waits on the Neo4j demo server.
- Fine-tuned jobs are queued first. The second GPU worker starts `WORKER_STAGGER_S` seconds after the first, so model loading does not compete for disk and the 4 vCPUs.

### Protocol
- `EVAL_LIMIT` rows from the test split, seed 3407, identical for every job.
- Fine-tuned (and cross-schema) jobs must log `Adapter verified` (history/027.md). Zero-shot jobs must not load an adapter.
- Greedy decoding; batch size and `max_new_tokens` from each config. Prompts are fitted to the training sequence budget (history/030.md).

### One-time setup per Kaggle account
1. Verify the account by phone (GPU and internet).
2. **Settings > Accelerator > GPU T4 x2** (not P100). **Settings > Internet > On.**
3. **Add-ons > Secrets:** add `HF_TOKEN` (read access to the checkpoint repo, write access to upload results) and attach it.

### Running
- Use **Save Version > Save & Run All (Commit)**.
- Lane output is not streamed (interleaved streams are unreadable). A heartbeat prints every `HEARTBEAT_S` seconds with each worker's job, step, latest log line, GPU memory and queue sizes.
- Each finished job uploads to `final-eval/<eval_tag>/jobs/<job_id>/` in one commit. If the session ends early, commit a new version: finished jobs on the same commit and platform are reused.
- Logs and the results bundle are written to `/kaggle/working` and saved with the version; the bundle is also uploaded to `final-eval/<eval_tag>/thesis/<session>/`.

### Caveat
- Latency (`s/item`) for jobs run here is measured with two predictions sharing 4 vCPUs. Each job records `platform` and `concurrent_gpu_workers`.

## Re-run plan (history/030.md)

The thesis bundle must come from **one commit and one platform**. Set `GIT_REF` to a tag, keep `EVAL_TAG_SUFFIX = 'r030'` (a fresh Hub namespace) and leave `REUSE_SAME_COMMIT_ONLY` / `REUSE_SAME_PLATFORM_ONLY` on.

Jobs that must run under that commit on Kaggle (12 GPU jobs):

| Run | fine-tuned | zero-shot | cross-schema |
|---|---|---|---|
| gemma-4-e2b-enhanced | yes (previously run on Colab) | yes | yes (exact-match prompts) |
| gemma-4-e2b-exact-match | yes (previously run on Colab) | yes | - |
| qwen3.5-2b-enhanced | yes | yes | yes (exact-match prompts) |
| qwen3.5-2b-exact-match | yes | yes | - |
| granite-4.2-3b-enhanced | yes | yes | - |

Every earlier job is invalidated for three reasons: inference prompts are now trimmed to the training budget, Neo4j outages were previously cached as query failures, and earlier jobs mixed commits and platforms. No retraining is needed.

In [ ]:
# 1. Final evaluation configuration (edit, then Save Version > Save & Run All)
import time

PIPELINE_T0 = time.time()  # Session time budget is measured from here

HUB_REPO = '2l3c34o1/text2cypher-checkpoints'
GITHUB_REPO = 'https://github.com/Cxmrykk/capstone.git'
GIT_REF = ''                  # Pin for thesis runs (tag or commit, e.g. 'thesis-r030'); blank is warned and flagged
EVAL_TAG_SUFFIX = 'r030'      # Fresh Hub namespace: outputs from before history/030.md are never reused
REUSE_SAME_COMMIT_ONLY = True     # Reuse a Hub job only if it ran on this exact commit
REUSE_SAME_PLATFORM_ONLY = True   # ...and on this platform
RUN_CROSS_SCHEMA = True       # Enhanced-trained adapter scored with exact-match prompts
DEAD_DB_POLICY = 'flag'       # 'flag' or 'exclude' databases whose gold queries all fail
EQUIV_MARGIN = 0.05           # TOST margin on execution-accuracy differences (absolute)
PRUNED_RATIO_THRESHOLD = 0.95 # Rows where the filter kept < 95% of the schema characters

# Experiments trained on Kaggle: (model, schema variant, config).
# Each gets one fine-tuned job and, if RUN_ZERO_SHOT, one zero-shot job.
EXPERIMENTS = [
    ('gemma-4-e2b', 'enhanced', 'configs/gemma-4-e2b.yaml'),
    ('gemma-4-e2b', 'pruned', 'configs/gemma-4-e2b-pruned.yaml'),
    ('qwen3.5-2b', 'enhanced', 'configs/qwen3.5-2b.yaml'),
    ('qwen3.5-2b', 'pruned', 'configs/qwen3.5-2b-pruned.yaml'),
    ('granite-4.2-3b', 'enhanced', 'configs/granite-4.2-3b.yaml'),
]

GPU_IDS = ['0', '1']          # One prediction worker per GPU (T4 x2)
N_EVAL_WORKERS = 2            # Concurrent `app.py evaluate` processes (CPU + Neo4j network)
WORKER_STAGGER_S = 60         # Delay before the second GPU worker starts

SPLIT = 'test'
EVAL_LIMIT = 500              # None = full split
RUN_ZERO_SHOT = True          # Base model without adapter, same prompts and rows
RUN_EXECUTION = True          # Execution accuracy against the Neo4j Labs demo server
RUN_SYNTAX_CHECK = True       # EXPLAIN-based syntax/schema validity
SKIP_COMPLETED_JOBS = True    # Reuse job outputs already on the Hub (same commit and platform only)

SESSION_BUDGET_MINUTES = 720  # Kaggle GPU session cap (12 h)
RESERVE_MINUTES = 60          # Kept for aggregation, figures and uploads at the end
MAX_JOB_MINUTES = 60          # A GPU worker does not start a job unless this much time remains before the reserve

BOOTSTRAP_SAMPLES = 2000      # Resamples for bootstrap confidence intervals
CI_LEVEL = 0.95
STATS_SEED = 20261014

RUN_GGUF_CPU_BENCH = False    # Optional: Q4_K_M GGUF on the Kaggle CPU via llama.cpp (runs after the GPU jobs)
GGUF_BENCH_LIMIT = 50         # A subset of the EVAL_LIMIT rows (same seed)

UPLOAD_TO_HUB = True          # Per-job outputs and the final results bundle
HEARTBEAT_S = 300

# Optional reference numbers copied by hand from published papers (e.g. Ozsoy et al., 2025).
# They are kept in a separate table and never mixed with measured results. Example entry:
# {'source': 'Ozsoy et al. (2025)', 'model': 'GPT-4o (zero-shot)', 'metric': 'google_bleu',
#  'value': <number from the paper>, 'notes': 'figure/table reference, different sample and protocol'}
PUBLISHED_BASELINES = []

NTFY_TOPIC = 'cxmrykk'        # Blank disables push notifications
NTFY_SERVER = 'https://ntfy.sh'

SCRATCH_DIR = '/kaggle/temp'  # Repo, models, artifacts (not saved; falls back to /tmp)
OUTPUT_DIR = '/kaggle/working'  # Logs and the results bundle (saved with the notebook version)

if len(GPU_IDS) != len(set(GPU_IDS)):
    raise ValueError('GPU_IDS must not repeat a GPU.')
print('GPU workers  : ' + ', '.join('GPU ' + g for g in GPU_IDS))
print('Eval workers : ' + str(N_EVAL_WORKERS))
print('Eval limit   : ' + str(EVAL_LIMIT))
print('Budget       : ' + str(SESSION_BUDGET_MINUTES) + ' min (reserve ' + str(RESERVE_MINUTES)
      + ' min, no new job within ' + str(MAX_JOB_MINUTES) + ' min of the reserve)')

In [ ]:
# 2. Platform checks, helpers, notifications and Hub access
import atexit
import codecs
import datetime
import json
import os
import platform
import queue
import shutil
import signal
import subprocess
import sys
import threading
import traceback
import urllib.request
from pathlib import Path

CR, LF = chr(13), chr(10)
PLATFORM = 'kaggle'  # The Colab notebook uses 'colab'

# --------------------------------------------------------------------------- #
# Platform checks (fail fast, before anything slow runs)
# --------------------------------------------------------------------------- #
ON_KAGGLE = Path('/kaggle').is_dir() or bool(os.environ.get('KAGGLE_KERNEL_RUN_TYPE'))
if not ON_KAGGLE:
    raise RuntimeError('This notebook targets Kaggle. On Google Colab use final-evaluation.ipynb.')
RUN_TYPE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'unknown')  # Interactive | Batch (commit)


def _query_gpus():
    """Returns {index: name} for physical GPUs (nvidia-smi ignores CUDA_VISIBLE_DEVICES)."""
    try:
        out = subprocess.run(['nvidia-smi', '--query-gpu=index,name', '--format=csv,noheader'],
                             capture_output=True, text=True, timeout=30).stdout
    except Exception:
        return {}
    gpus = {}
    for line in out.splitlines():
        if ',' in line:
            idx, name = line.split(',', 1)
            gpus[idx.strip()] = name.strip()
    return gpus


GPUS = _query_gpus()
if not GPUS:
    raise RuntimeError('No GPU visible. Settings > Accelerator > GPU T4 x2 (phone-verified account).')
for gid in GPU_IDS:
    name = GPUS.get(gid)
    if name is None:
        raise RuntimeError('GPU ' + gid + ' does not exist (found ' + str(sorted(GPUS)) + '). '
                           'Two workers need the GPU T4 x2 accelerator, or set GPU_IDS = ["0"].')
    if 'P100' in name:
        raise RuntimeError('Tesla P100 (Pascal) selected. Switch the accelerator to GPU T4 x2.')
    if 'T4' not in name:
        print('WARNING: expected a Tesla T4 on GPU ' + gid + ', found ' + name + '. Accuracy is unaffected, '
              'but latency figures will not match the T4 used for training.')
GPU_NAME = GPUS[GPU_IDS[0]]
GPU_LABEL = str(len(GPU_IDS)) + 'x ' + GPU_NAME if len(set(GPUS[g] for g in GPU_IDS)) == 1 \
    else ' + '.join(GPUS[g] for g in GPU_IDS)

# Shared steps (doctor, neo4j checks) see one GPU. Every predict step overrides
# CUDA_VISIBLE_DEVICES with its worker's GPU, so no process ever sees two devices.
os.environ['CUDA_VISIBLE_DEVICES'] = GPU_IDS[0]


def _internet_ok(url='https://huggingface.co', timeout=15):
    try:
        with urllib.request.urlopen(url, timeout=timeout) as resp:
            return resp.status < 500
    except Exception:
        return False


if not _internet_ok():
    raise RuntimeError('No internet access. Settings > Internet > On (phone-verified account).')


def _writable_dir(path):
    try:
        p = Path(path)
        p.mkdir(parents=True, exist_ok=True)
        probe = p / '.t2c-write-test'
        probe.write_text('ok', encoding='utf-8')
        probe.unlink()
        return True
    except Exception:
        return False


def _free_gb(path):
    try:
        return shutil.disk_usage(path).free / 1e9
    except Exception:
        return float('nan')


WORK_ROOT = Path(SCRATCH_DIR) if _writable_dir(SCRATCH_DIR) else Path('/tmp')
REPO_DIR = WORK_ROOT / 'capstone'
SESSION = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d-%H%M%S') + '-kaggle'
LOG_DIR = Path(OUTPUT_DIR) / 'final-eval-logs' / SESSION
LOG_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------------------- #
# Secrets and environment
# --------------------------------------------------------------------------- #
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception as exc:
    raise RuntimeError('Could not read the HF_TOKEN Kaggle secret (' + type(exc).__name__ + '). '
                       'Add-ons > Secrets: add HF_TOKEN and attach it. The adapters live in a private repo.') from exc

os.environ['T2C_HUB_REPO'] = HUB_REPO
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# --------------------------------------------------------------------------- #
# Shared state
# --------------------------------------------------------------------------- #
STATUS = {'session': SESSION, 'platform': PLATFORM, 'run_type': RUN_TYPE, 'gpus': GPUS,
          'jobs': {}, 'outcome': None}
WORKERS = {}                   # scope -> {'job', 'step', 'started', 'last_line'}
_state_lock = threading.RLock()
STOP_EVENT = threading.Event()
ACTIVE_PROCS = {}
_procs_lock = threading.Lock()

# --------------------------------------------------------------------------- #
# ntfy push notifications (best-effort; the topic is public, so no secrets)
# --------------------------------------------------------------------------- #
NTFY_URL = (NTFY_SERVER.rstrip('/') + '/' + NTFY_TOPIC.strip()) if NTFY_TOPIC.strip() else None


def notify(title, message, priority='default', tags=()):
    if not NTFY_URL:
        return False
    headers = {'Title': ('T2C Kaggle final eval: ' + title).encode('ascii', 'replace').decode('ascii'),
               'Priority': str(priority)}
    if tags:
        headers['Tags'] = ','.join(tags)
    body = message if len(message) <= 3000 else '...' + message[-3000:]
    req = urllib.request.Request(NTFY_URL, data=body.encode('utf-8'), headers=headers, method='POST')
    try:
        with urllib.request.urlopen(req, timeout=10) as resp:
            resp.read()
        return True
    except Exception as exc:
        print('[ntfy] Notification failed: ' + str(exc), file=sys.stderr)
        return False


# --------------------------------------------------------------------------- #
# Step runner: thread-safe, writes a log file (tqdm redraws collapsed)
# --------------------------------------------------------------------------- #
def _clean_lines(buf, text):
    text = buf + text.replace(CR + LF, LF)
    parts = text.split(LF)
    return [p.split(CR)[-1] for p in parts[:-1]], parts[-1].split(CR)[-1]


def _interrupt_proc(proc):
    try:
        os.killpg(proc.pid, signal.SIGINT)
        return proc.wait(timeout=60)
    except Exception:
        proc.kill()
        return proc.wait()


def _stop_all_procs():
    with _procs_lock:
        procs = list(ACTIVE_PROCS.values())
    for proc in procs:
        try:
            os.killpg(proc.pid, signal.SIGINT)
        except Exception:
            pass


def run_step(name, cmd, log_path=None, allow_fail=False, env=None, cwd=None, echo=True, scope='main'):
    """Runs a shell command, writing its output to a log file.

    Main-thread steps stream to the cell (echo=True). Worker steps use echo=False;
    the heartbeat shows their latest log line instead.
    """
    if STOP_EVENT.is_set():
        raise KeyboardInterrupt('Stop requested before step ' + name + '.')
    log_path = Path(log_path) if log_path else LOG_DIR / (scope + '-' + name + '.log')
    log_path.parent.mkdir(parents=True, exist_ok=True)
    workdir = cwd or (str(REPO_DIR) if REPO_DIR.exists() else str(WORK_ROOT))
    with _state_lock:
        WORKERS.setdefault(scope, {}).update(step=name, started=time.time(), last_line='')
    if echo:
        print('=' * 72)
        print('[' + scope + '] ' + name + ': ' + cmd)
        print('    log: ' + str(log_path))
        print('=' * 72, flush=True)
    else:
        print('[' + scope + '] ' + name + ' started (log: ' + str(log_path) + ')', flush=True)

    t0 = time.time()
    buf = ''
    interrupted = False
    decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
    proc_key = (scope, name, t0)
    with open(log_path, 'w', encoding='utf-8') as fh:
        fh.write('$ ' + cmd + LF + '# cwd: ' + workdir + LF + LF)
        fh.flush()
        proc = subprocess.Popen(cmd, shell=True, cwd=workdir, env={**os.environ, **(env or {})},
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, start_new_session=True)
        with _procs_lock:
            ACTIVE_PROCS[proc_key] = proc
        try:
            while True:
                chunk = proc.stdout.read1(8192)
                if not chunk:
                    break
                text = decoder.decode(chunk)
                if echo:
                    sys.stdout.write(text)
                    sys.stdout.flush()
                lines, buf = _clean_lines(buf, text)
                for line in lines:
                    fh.write(line + LF)
                fh.flush()
                current = buf.strip() or (lines[-1].strip() if lines else '')
                if current:
                    with _state_lock:
                        WORKERS[scope]['last_line'] = current
            rc = proc.wait()
        except KeyboardInterrupt:
            interrupted = True
            rc = _interrupt_proc(proc)
        finally:
            with _procs_lock:
                ACTIVE_PROCS.pop(proc_key, None)
        if buf:
            fh.write(buf + LF)
        elapsed = time.time() - t0
        fh.write(LF + '# exit code: ' + str(rc) + LF + '# elapsed: ' + format(elapsed, '.1f') + 's' + LF)

    with _state_lock:
        WORKERS[scope].update(step=None, started=None)
    print('[' + scope + '] ' + name + ': exit ' + str(rc) + ' after ' + format(elapsed / 60, '.1f') + ' min',
          flush=True)
    if interrupted or (STOP_EVENT.is_set() and rc != 0):
        raise KeyboardInterrupt('Step ' + name + ' (' + scope + ') interrupted.')
    if rc != 0 and not allow_fail:
        raise RuntimeError('Step ' + name + ' failed with exit code ' + str(rc) + '. See ' + str(log_path))
    return rc


# --------------------------------------------------------------------------- #
# Hub access (uploads are serialised: concurrent commits to one repo conflict)
# --------------------------------------------------------------------------- #
_HUB = {'files': None}
_hub_lock = threading.Lock()


def hub_api():
    from huggingface_hub import HfApi
    return HfApi(token=os.environ['HF_TOKEN'])


def hub_files(refresh=False):
    if _HUB['files'] is None or refresh:
        _HUB['files'] = set(hub_api().list_repo_files(HUB_REPO, repo_type='model'))
    return _HUB['files']


def hub_download_prefix(prefix, dest):
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=HUB_REPO, repo_type='model', token=os.environ['HF_TOKEN'],
                      allow_patterns=[prefix + '/*'], local_dir=str(dest))
    return Path(dest) / prefix


def hub_download_file(path_in_repo):
    from huggingface_hub import hf_hub_download
    return Path(hf_hub_download(repo_id=HUB_REPO, filename=path_in_repo, repo_type='model',
                                token=os.environ['HF_TOKEN']))


def hub_upload_folder(local_dir, path_in_repo, message):
    """Uploads a folder in ONE commit, with retries. Never raises."""
    if not UPLOAD_TO_HUB:
        return False
    for attempt in range(3):
        try:
            with _hub_lock:
                hub_api().upload_folder(folder_path=str(local_dir), path_in_repo=path_in_repo,
                                        repo_id=HUB_REPO, repo_type='model', commit_message=message)
            return True
        except Exception as exc:
            print('[hub] Upload of ' + path_in_repo + ' failed (attempt ' + str(attempt + 1) + '/3): ' + str(exc),
                  flush=True)
            time.sleep(30 * (attempt + 1))
    return False


# --------------------------------------------------------------------------- #
# Crash hooks
# --------------------------------------------------------------------------- #
def _notify_unfinished(reason):
    if STATUS.get('outcome') is not None:
        return
    STATUS['outcome'] = 'runtime-ended'
    done = [k for k, v in STATUS['jobs'].items() if v == 'completed']
    notify('runtime ended', reason + ' before the final cell. Session ' + SESSION + '. Completed jobs: '
           + (', '.join(done) or 'none') + '. Commit a new version: finished jobs are reused from the Hub.',
           priority='urgent', tags=['skull'])


def _on_sigterm(signum, frame):
    _notify_unfinished('The Kaggle session was shut down (SIGTERM)')
    STOP_EVENT.set()
    _stop_all_procs()
    prev = globals().get('_PREV_SIGTERM')
    if callable(prev):
        prev(signum, frame)
    else:
        signal.signal(signal.SIGTERM, signal.SIG_DFL)
        os.kill(os.getpid(), signal.SIGTERM)


def _on_cell_exception(shell, etype, evalue, tb, tb_offset=None):
    notify('notebook error', etype.__name__ + ': ' + str(evalue) + LF + 'Session ' + SESSION,
           priority='urgent', tags=['rotating_light'])
    shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)


if not globals().get('_HOOKS_INSTALLED'):
    _PREV_SIGTERM = signal.getsignal(signal.SIGTERM)
    signal.signal(signal.SIGTERM, _on_sigterm)
    atexit.register(lambda: _notify_unfinished('The notebook kernel exited'))
    _HOOKS_INSTALLED = True
get_ipython().set_custom_exc((Exception,), _on_cell_exception)

print('Platform   : Kaggle (' + RUN_TYPE + ' session)')
print('GPUs       : ' + str(GPUS))
print('Work root  : ' + str(WORK_ROOT) + ' (' + format(_free_gb(WORK_ROOT), '.0f') + ' GB free, not saved)')
print('Output dir : ' + OUTPUT_DIR + ' (' + format(_free_gb(OUTPUT_DIR), '.0f') + ' GB free, saved)')
print('Session    : ' + SESSION)
print('Logs       : ' + str(LOG_DIR))
notify('started', 'Session ' + SESSION + ' (' + RUN_TYPE + ') on ' + GPU_LABEL + '.', priority='low', tags=['rocket'])

In [ ]:
# 3. Clone the repository and pin the code version
if (REPO_DIR / '.git').exists():
    run_step('git-fetch', 'git -C ' + str(REPO_DIR) + ' fetch --all --tags --prune', cwd=str(WORK_ROOT),
             allow_fail=True)
    if not GIT_REF:
        run_step('git-pull', 'git -C ' + str(REPO_DIR) + ' pull --ff-only', cwd=str(WORK_ROOT), allow_fail=True)
else:
    run_step('git-clone', 'git clone ' + GITHUB_REPO + ' ' + str(REPO_DIR), cwd=str(WORK_ROOT))
if GIT_REF:
    run_step('git-checkout', 'git -C ' + str(REPO_DIR) + ' checkout ' + GIT_REF, cwd=str(WORK_ROOT))

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))


def _git(*args):
    return subprocess.run(['git', *args], cwd=REPO_DIR, capture_output=True, text=True).stdout.strip()


GIT_COMMIT = _git('rev-parse', 'HEAD')
GIT_DIRTY = bool(_git('status', '--porcelain'))
if not GIT_REF:
    print('WARNING: GIT_REF is blank, so this bundle is not pinned to a tag. Set GIT_REF for thesis runs.')

# Predictions made before history/027.md silently evaluated the base Qwen model.
# Refuse to run on code that lacks the verified adapter loader.
loader_src = (REPO_DIR / 'src' / 'training' / 'model_loader.py').read_text(encoding='utf-8')
if 'def attach_adapter' not in loader_src or 'Adapter verified' not in loader_src:
    raise RuntimeError('This checkout lacks the verified adapter loader (history/027.md). '
                       'Pull a newer commit or clear GIT_REF.')
dataset_src = (REPO_DIR / 'src' / 'data' / 'dataset.py').read_text(encoding='utf-8')
neo4j_src = (REPO_DIR / 'src' / 'evaluation' / 'neo4j_client.py').read_text(encoding='utf-8')
if 'def fit_prompt_to_budget' not in dataset_src or 'connection_failed' not in neo4j_src:
    raise RuntimeError('This checkout lacks the history/030.md changes (prompt budget, Neo4j fixes).')
print('Commit : ' + GIT_COMMIT + (' (uncommitted changes)' if GIT_DIRTY else ''))

In [ ]:
# 4. Install dependencies and download the dataset and base models (sequential downloads)
# Unsloth is not needed: inference loads adapters through Transformers + PEFT.
run_step('bootstrap', 'bash scripts/colab_bootstrap.sh', env={'INSTALL_UNSLOTH': '0'})

DOWNLOAD_TARGETS = {'gemma-4-e2b': 'gemma', 'qwen3.5-2b': 'qwen', 'granite-4.2-3b': 'granite'}
for target in dict.fromkeys(DOWNLOAD_TARGETS[m] for m, _v, _c in EXPERIMENTS):
    run_step('download-' + target, 'bash download_data.sh ' + target)

run_step('doctor', 'python app.py doctor', allow_fail=True)
print('Free disk in work root: ' + format(_free_gb(WORK_ROOT), '.0f') + ' GB')

In [ ]:
# 5. Check the Neo4j Labs demo server (execution accuracy depends on it)
if RUN_EXECUTION:
    rc = run_step('neo4j-check', 'python app.py neo4j check --alias neo4jlabs_demo_db_movies', allow_fail=True)
    if rc != 0:
        raise RuntimeError('The Neo4j Labs demo server is unreachable from this Kaggle session (outbound Bolt on '
                           'port 7687). Retry later, or set RUN_EXECUTION = False to report text metrics only.')
    run_step('neo4j-aliases', 'python app.py neo4j aliases', allow_fail=True, echo=False)
else:
    print('RUN_EXECUTION is False: execution accuracy will not be measured.')

In [ ]:
# 6. Resolve runs, pull the final adapters from the Hub and check their provenance
import yaml
from src.config import load_config

EXP = []
for model, variant, config in EXPERIMENTS:
    raw = yaml.safe_load(Path(config).read_text(encoding='utf-8'))
    cfg = load_config(config)
    run_name = raw['run_name']
    EXP.append({
        'model': model, 'variant': variant, 'config': config, 'run_name': run_name,
        'schema_mode': cfg.data.schema_mode, 'seed': cfg.data.seed,
        'fingerprint_now': cfg.fingerprint(),
        'adapter_dir': REPO_DIR / 'artifacts' / 'runs' / run_name / 'final',
    })

seeds = sorted({e['seed'] for e in EXP})
if len(seeds) != 1:
    raise RuntimeError('Configs use different data seeds ' + str(seeds) + '; jobs would be scored on different rows.')
SAMPLE_SEED = seeds[0]
LIMIT_LABEL = str(EVAL_LIMIT) if EVAL_LIMIT else 'all'
EVAL_TAG = SPLIT + '-n' + LIMIT_LABEL + '-seed' + str(SAMPLE_SEED) + ('-' + EVAL_TAG_SUFFIX if EVAL_TAG_SUFFIX else '')
# predict falls back to data.max_eval_samples (1000) when no limit is passed, so
# the full split needs an explicit limit larger than the split.
PREDICT_LIMIT = EVAL_LIMIT if EVAL_LIMIT else 10 ** 7
HUB_EVAL_PREFIX = 'final-eval/' + EVAL_TAG
EVAL_DIR = REPO_DIR / 'artifacts' / 'final-eval' / EVAL_TAG
HUB_CACHE = REPO_DIR / 'artifacts' / 'hub_cache'
EVAL_DIR.mkdir(parents=True, exist_ok=True)
print('Evaluation tag: ' + EVAL_TAG)

for e in EXP:
    rn = e['run_name']
    adir = e['adapter_dir']
    if not (adir / 'adapter_config.json').exists():
        rc = run_step('pull-' + rn, 'python app.py checkpoint pull --config ' + e['config'] + ' --step final',
                      allow_fail=True)
        if rc not in (0, 1):
            raise RuntimeError(rn + ': fetching the final adapter failed with exit ' + str(rc)
                               + ' (Hub error, not "not found"). Rerun when the Hub is reachable.')
    e['adapter_ok'] = (adir / 'adapter_config.json').exists() and any(adir.glob('adapter_model.*'))
    meta_path = adir / 'run_metadata.json'
    e['train_meta'] = json.loads(meta_path.read_text(encoding='utf-8')) if meta_path.exists() else {}
    fp = e['train_meta'].get('fingerprint')
    e['fingerprint_trained'] = fp
    e['fingerprint_match'] = (fp == e['fingerprint_now']) if fp else None

print()
print('Run'.ljust(30) + 'Adapter'.ljust(9) + 'Step'.ljust(7) + 'Fingerprint')
for e in EXP:
    tm = e['train_meta']
    fp_state = {True: 'match', False: 'DIFFERS', None: 'n/a'}[e['fingerprint_match']]
    print(e['run_name'].ljust(30) + ('OK' if e['adapter_ok'] else 'MISSING').ljust(9)
          + str(tm.get('global_step', '-')).ljust(7) + fp_state)

if any(e['fingerprint_match'] is False for e in EXP):
    print()
    print('NOTE: a differing fingerprint usually means a config file or RunConfig field changed after '
          'training. Adapter loading is still verified tensor-by-tensor in every fine-tuned job.')
missing = [e['run_name'] for e in EXP if not e['adapter_ok']]
if missing:
    print()
    print('WARNING: no final adapter on the Hub for: ' + ', '.join(missing) + '. Their adapter jobs are recorded as '
          'missing_adapter and fail an integrity check.')

# Which models have no chat template (prompt falls back to plain text + BOS).
from src.training.model_loader import load_tokenizer
PLAIN_TEXT_PROMPT = {}
for e in EXP:
    if e['model'] not in PLAIN_TEXT_PROMPT:
        tok = load_tokenizer(load_config(e['config']))
        PLAIN_TEXT_PROMPT[e['model']] = not bool(getattr(tok, 'chat_template', None))
    e['plain_text_prompt'] = PLAIN_TEXT_PROMPT[e['model']]
print('Plain-text prompt (no chat template): ' + str(PLAIN_TEXT_PROMPT))

In [ ]:
# 7. Run the evaluation jobs on both GPUs:
#    GPU workers: predict -> verify adapter -> hand off
#    Eval workers: evaluate -> record -> upload (one commit per job)
ADAPTER_CONDITIONS = ('fine_tuned', 'cross_schema')
CONDITION_ORDER = {'fine_tuned': 0, 'cross_schema': 1, 'zero_shot': 2}

# The expected job list comes from EXPERIMENTS alone. A job whose adapter is missing
# stays expected: it is recorded as not run and fails an integrity check (history/030.md).
JOBS = []
for e in EXP:
    for condition in ['fine_tuned'] + (['zero_shot'] if RUN_ZERO_SHOT else []):
        JOBS.append({**e, 'condition': condition, 'backend': 'hf',
                     'prompt_schema_mode': e['schema_mode'],
                     'train_schema_mode': e['schema_mode'] if condition == 'fine_tuned' else None,
                     'job_id': e['run_name'] + '__' + condition})
if RUN_CROSS_SCHEMA:
    # Enhanced-trained adapter scored with exact-match prompts: separates the train-time
    # effect of schema filtering from its inference-time effect.
    by_variant = {(e['model'], e['variant']): e for e in EXP}
    for (m, v), e in by_variant.items():
        pruned = by_variant.get((m, 'pruned'))
        if v != 'enhanced' or pruned is None:
            continue
        JOBS.append({**e, 'condition': 'cross_schema', 'backend': 'hf',
                     'schema_mode': pruned['schema_mode'],
                     'prompt_schema_mode': pruned['schema_mode'],
                     'train_schema_mode': e['schema_mode'],
                     'job_id': e['run_name'] + '__fine_tuned__' + pruned['schema_mode'] + '_prompts'})
JOBS.sort(key=lambda j: CONDITION_ORDER.get(j['condition'], 9))

JOB_RESULTS = {}
_results_lock = threading.Lock()
SKIPPED_FOR_TIME = []


def _set_result(jid, rec, status):
    with _results_lock:
        JOB_RESULTS[jid] = rec
        STATUS['jobs'][jid] = status


def _job_record(job, **extra):
    keep = ('job_id', 'model', 'variant', 'config', 'run_name', 'schema_mode', 'condition', 'backend',
            'prompt_schema_mode', 'train_schema_mode', 'plain_text_prompt',
            'fingerprint_now', 'fingerprint_trained', 'fingerprint_match')
    rec = {k: job.get(k) for k in keep}
    rec.update({'eval_tag': EVAL_TAG, 'split': SPLIT, 'limit': EVAL_LIMIT, 'sample_seed': SAMPLE_SEED,
                'git_ref': GIT_REF, 'git_commit': GIT_COMMIT, 'git_dirty': GIT_DIRTY, 'gpu': GPU_LABEL,
                'platform': PLATFORM, 'session': SESSION, 'execution': RUN_EXECUTION,
                'syntax_check': RUN_SYNTAX_CHECK, 'dead_db_policy': DEAD_DB_POLICY,
                'concurrent_gpu_workers': len(GPU_IDS)})
    if job['condition'] in ADAPTER_CONDITIONS:
        rec['adapter'] = str(job['adapter_dir'])
        rec['train_global_step'] = (job.get('train_meta') or {}).get('global_step')
    if job['backend'] == 'llamacpp':
        rec['gguf'] = job.get('gguf_repo_path')
        rec['limit'] = GGUF_BENCH_LIMIT
    rec.update(extra)
    return rec


def _reuse_from_hub(job, jdir):
    prefix = HUB_EVAL_PREFIX + '/jobs/' + job['job_id']
    if not SKIP_COMPLETED_JOBS or (prefix + '/job.json') not in hub_files():
        return None
    rec = json.loads(hub_download_file(prefix + '/job.json').read_text(encoding='utf-8'))
    if rec.get('status') != 'completed':
        return None
    origin = rec.get('platform') or 'colab'
    if REUSE_SAME_COMMIT_ONLY and rec.get('git_commit') != GIT_COMMIT:
        print('[' + job['job_id'] + '] Hub outputs come from commit ' + str(rec.get('git_commit'))[:10]
              + '; recomputing on ' + GIT_COMMIT[:10] + '.')
        return None
    if REUSE_SAME_PLATFORM_ONLY and origin != PLATFORM:
        print('[' + job['job_id'] + '] Hub outputs come from ' + origin + '; recomputing on ' + PLATFORM + '.')
        return None
    src = hub_download_prefix(prefix, HUB_CACHE)
    if jdir.exists():
        shutil.rmtree(jdir)
    shutil.copytree(src, jdir)
    if not (jdir / 'predictions.scored.jsonl').exists():
        return None
    return rec


def _time_left_for_job():
    elapsed_min = (time.time() - PIPELINE_T0) / 60
    return elapsed_min + MAX_JOB_MINUTES <= SESSION_BUDGET_MINUTES - RESERVE_MINUTES


def _predict(job, gpu, scope, echo=False):
    """Runs predict on one GPU and checks adapter provenance. Returns a context dict."""
    jid = job['job_id']
    jdir = EVAL_DIR / 'jobs' / jid
    if jdir.exists():
        shutil.rmtree(jdir)
    jdir.mkdir(parents=True)
    pred = jdir / 'predictions.jsonl'
    started = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')
    t0 = time.time()

    if job['backend'] == 'llamacpp':
        cmd = ('python app.py predict --config ' + job['config'] + ' --backend llamacpp --gguf '
               + str(job['gguf_local']) + ' --split ' + SPLIT + ' --limit ' + str(GGUF_BENCH_LIMIT)
               + ' --out ' + str(pred) + ' --tag ' + jid)
    else:
        cmd = ('python app.py predict --config ' + job['config'] + ' --backend hf --split ' + SPLIT
               + ' --limit ' + str(PREDICT_LIMIT) + ' --schema-mode ' + job['prompt_schema_mode']
               + ' --out ' + str(pred) + ' --tag ' + jid)
        if job['condition'] in ADAPTER_CONDITIONS:
            cmd += ' --adapter ' + str(job['adapter_dir'])
    run_step('predict-' + jid, cmd, log_path=jdir / 'predict.log', env={'CUDA_VISIBLE_DEVICES': gpu},
             echo=echo, scope=scope)

    # Adapter provenance (history/027.md): a fine-tuned score is only valid if the
    # verified loader confirmed every LoRA tensor was loaded.
    log_lines = (jdir / 'predict.log').read_text(encoding='utf-8', errors='replace').splitlines()
    verified = [ln.split('| ', 1)[-1] for ln in log_lines if 'Adapter verified:' in ln]
    remapped = [ln.split('| ', 1)[-1] for ln in log_lines if 'Remapped' in ln and 'adapter tensor' in ln]
    classes = [ln.split('Loaded model class:', 1)[-1].strip() for ln in log_lines if 'Loaded model class:' in ln]
    loaded_adapter = any('Loading LoRA adapter' in ln for ln in log_lines)
    if job['backend'] == 'hf' and job['condition'] in ADAPTER_CONDITIONS and not verified:
        raise RuntimeError(jid + ': the predict log has no Adapter verified line; refusing to score a '
                           'possibly unadapted model.')
    if job['condition'] == 'zero_shot' and loaded_adapter:
        raise RuntimeError(jid + ': an adapter was loaded for a zero-shot job.')

    return {'jdir': jdir, 'pred': pred, 'started': started, 't0': t0, 'predict_wall_s': round(time.time() - t0, 1),
            'gpu': gpu, 'verified': verified, 'remapped': remapped, 'classes': classes}


def _evaluate(job, ctx, scope, echo=False):
    """Scores a finished prediction file, writes job.json and uploads the job folder."""
    jid = job['job_id']
    jdir, pred = ctx['jdir'], ctx['pred']
    report = jdir / 'report.md'
    t_eval = time.time()
    flags = ((' --execute' if RUN_EXECUTION else '') + (' --validate-syntax' if RUN_SYNTAX_CHECK else '')
             + ' --dead-db-policy ' + DEAD_DB_POLICY)
    run_step('evaluate-' + jid, 'python app.py evaluate --predictions ' + str(pred) + flags + ' --out ' + str(report),
             log_path=jdir / 'evaluate.log', echo=echo, scope=scope)
    for required in ('predictions.scored.jsonl', 'report.json', 'predictions.meta.json'):
        if not (jdir / required).exists():
            raise RuntimeError(jid + ': missing ' + required + ' after evaluation.')

    meta = json.loads((jdir / 'predictions.meta.json').read_text(encoding='utf-8'))
    rec = _job_record(
        job, status='completed', started=ctx['started'],
        finished=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
        wall_s=round(time.time() - ctx['t0'], 1), predict_wall_s=ctx['predict_wall_s'],
        evaluate_wall_s=round(time.time() - t_eval, 1),
        gpu=GPUS.get(ctx['gpu'], GPU_NAME), gpu_index=ctx['gpu'],
        n_rows=meta.get('n_instances'), seconds_per_item=meta.get('seconds_per_item'),
        generation_seconds=meta.get('total_seconds'), mean_prompt_tokens=meta.get('mean_prompt_tokens'),
        batch_size=meta.get('batch_size'), max_new_tokens=meta.get('max_new_tokens'),
        temperature=meta.get('temperature'), adapter_verified=ctx['verified'],
        adapter_remapped=ctx['remapped'], model_class=ctx['classes'],
        n_schema_trimmed=meta.get('n_schema_trimmed'), n_prompt_over_budget=meta.get('n_prompt_over_budget'),
        prompt_reserve_tokens=meta.get('prompt_reserve_tokens'),
        gguf_merge_metadata=job.get('gguf_merge_metadata'), python=platform.python_version(),
    )
    (jdir / 'job.json').write_text(json.dumps(rec, indent=2, default=str), encoding='utf-8')
    rec['uploaded'] = hub_upload_folder(jdir, HUB_EVAL_PREFIX + '/jobs/' + jid, '[final-eval] ' + EVAL_TAG + ' ' + jid)
    if UPLOAD_TO_HUB and not rec['uploaded']:
        print('WARNING: ' + jid + ' finished but its upload failed; it is only kept in this session.', flush=True)
    return rec


# --------------------------------------------------------------------------- #
# Reuse finished jobs (main thread), queue the rest
# --------------------------------------------------------------------------- #
print('Planned jobs (' + str(len(JOBS)) + '):')
for job in JOBS:
    print('  ' + job['job_id'])
print()

GPU_QUEUE = queue.Queue()
EVAL_QUEUE = queue.Queue()
for job in JOBS:
    jid = job['job_id']
    if job['condition'] in ADAPTER_CONDITIONS and not job['adapter_ok']:
        print('[' + jid + '] Expected job cannot run: no final adapter for ' + job['run_name'] + ' on the Hub.')
        _set_result(jid, _job_record(job, status='missing_adapter', error='no final adapter on the Hub'),
                    'missing_adapter')
        continue
    try:
        reused = _reuse_from_hub(job, EVAL_DIR / 'jobs' / jid)
    except Exception as exc:
        print('[' + jid + '] Could not check the Hub for completed outputs (' + str(exc) + '); recomputing.')
        reused = None
    if reused is not None:
        origin = reused.get('platform') or 'colab'
        print('[' + jid + '] Reusing completed outputs from the Hub (' + origin + ', commit '
              + str(reused.get('git_commit'))[:10] + ').')
        _set_result(jid, reused, 'completed')
    else:
        GPU_QUEUE.put(job)
print('Jobs to compute: ' + str(GPU_QUEUE.qsize()))


# --------------------------------------------------------------------------- #
# Workers
# --------------------------------------------------------------------------- #
def gpu_worker(gpu, stagger_s):
    scope = 'gpu' + gpu
    with _state_lock:
        WORKERS.setdefault(scope, {}).update(job=None, step=None, started=None, last_line='waiting to start')
    delay = stagger_s
    while delay > 0 and not STOP_EVENT.is_set():
        time.sleep(min(5, delay))
        delay -= 5
    while not STOP_EVENT.is_set():
        try:
            job = GPU_QUEUE.get_nowait()
        except queue.Empty:
            break
        jid = job['job_id']
        if not _time_left_for_job():
            with _results_lock:
                SKIPPED_FOR_TIME.append(jid)
            print('[' + scope + '] Not starting ' + jid + ': too little session time left.', flush=True)
            continue
        with _state_lock:
            WORKERS[scope]['job'] = jid
        try:
            ctx = _predict(job, gpu, scope)
            EVAL_QUEUE.put((job, ctx))
        except KeyboardInterrupt:
            _set_result(jid, _job_record(job, status='interrupted'), 'interrupted')
            break
        except Exception as exc:
            _set_result(jid, _job_record(job, status='failed', error=str(exc), gpu_index=gpu), 'failed')
            print('[' + scope + '] ' + jid + ' FAILED: ' + str(exc) + LF + traceback.format_exc(), flush=True)
            notify('job failed', jid + ' (predict, GPU ' + gpu + '): ' + str(exc)[-1500:], priority='high',
                   tags=['warning'])
    with _state_lock:
        WORKERS[scope].update(job=None, last_line='done')


def eval_worker(idx):
    scope = 'eval' + str(idx)
    with _state_lock:
        WORKERS.setdefault(scope, {}).update(job=None, step=None, started=None, last_line='idle')
    while True:
        item = EVAL_QUEUE.get()
        if item is None:
            break
        job, ctx = item
        jid = job['job_id']
        with _state_lock:
            WORKERS[scope]['job'] = jid
        try:
            rec = _evaluate(job, ctx, scope)
            _set_result(jid, rec, 'completed')
            print('[' + scope + '] ' + jid + ' completed.', flush=True)
        except KeyboardInterrupt:
            _set_result(jid, _job_record(job, status='interrupted'), 'interrupted')
        except Exception as exc:
            _set_result(jid, _job_record(job, status='failed', error=str(exc), gpu_index=ctx.get('gpu')), 'failed')
            print('[' + scope + '] ' + jid + ' FAILED: ' + str(exc) + LF + traceback.format_exc(), flush=True)
            notify('job failed', jid + ' (evaluate): ' + str(exc)[-1500:], priority='high', tags=['warning'])
        with _state_lock:
            WORKERS[scope].update(job=None, last_line='idle')


def _gpu_usage():
    try:
        out = subprocess.run(['nvidia-smi', '--query-gpu=index,memory.used,memory.total,utilization.gpu',
                              '--format=csv,noheader,nounits'], capture_output=True, text=True, timeout=20).stdout
    except Exception:
        return {}
    usage = {}
    for line in out.splitlines():
        parts = [p.strip() for p in line.split(',')]
        if len(parts) == 4:
            idx, used, total, util = parts
            try:
                usage[idx] = (format(float(used) / 1024, '.1f') + '/' + format(float(total) / 1024, '.1f')
                              + ' GiB ' + util + '%')
            except ValueError:
                pass
    return usage


def _ram_available_gb():
    try:
        for line in Path('/proc/meminfo').read_text().splitlines():
            if line.startswith('MemAvailable:'):
                return int(line.split()[1]) / 1024 ** 2
    except Exception:
        pass
    return float('nan')


def heartbeat():
    usage = _gpu_usage()
    now = datetime.datetime.now(datetime.timezone.utc).strftime('%H:%M:%S')
    elapsed = (time.time() - PIPELINE_T0) / 60
    with _results_lock:
        n_done = sum(1 for v in STATUS['jobs'].values() if v == 'completed')
    lines = ['--- ' + now + ' UTC | session ' + format(elapsed, '.0f') + '/' + str(SESSION_BUDGET_MINUTES)
             + ' min | completed ' + str(n_done) + '/' + str(len(JOBS)) + ' | GPU queue ' + str(GPU_QUEUE.qsize())
             + ' | eval queue ' + str(EVAL_QUEUE.qsize()) + ' | RAM available '
             + format(_ram_available_gb(), '.1f') + ' GB']
    for gid in GPU_IDS:
        lines.append('  GPU' + gid + ' ' + usage.get(gid, '?'))
    with _state_lock:
        snapshot = {k: dict(v) for k, v in WORKERS.items() if k != 'main'}
    for scope in sorted(snapshot):
        st = snapshot[scope]
        if st.get('step'):
            started = st.get('started') or time.time()
            state = st['step'] + ' ' + format((time.time() - started) / 60, '.0f') + ' min'
        else:
            state = 'idle'
        last = (st.get('last_line') or '')[-110:]
        lines.append('  [' + scope + '] ' + state + ' | ' + last)
    print(LF.join(lines), flush=True)


# --------------------------------------------------------------------------- #
# Run
# --------------------------------------------------------------------------- #
eval_threads = [threading.Thread(target=eval_worker, args=(i,), name='eval-' + str(i), daemon=True)
                for i in range(N_EVAL_WORKERS)]
gpu_threads = [threading.Thread(target=gpu_worker, args=(gid, i * WORKER_STAGGER_S), name='gpu-' + gid, daemon=True)
               for i, gid in enumerate(GPU_IDS)]
for t in eval_threads + gpu_threads:
    t.start()

try:
    last_beat = time.time()
    while any(t.is_alive() for t in gpu_threads):
        time.sleep(5)
        if time.time() - last_beat >= HEARTBEAT_S:
            heartbeat()
            last_beat = time.time()
    for _ in eval_threads:
        EVAL_QUEUE.put(None)
    while any(t.is_alive() for t in eval_threads):
        time.sleep(5)
        if time.time() - last_beat >= HEARTBEAT_S:
            heartbeat()
            last_beat = time.time()
except KeyboardInterrupt:
    print('Interrupt received: stopping all workers...', flush=True)
    STOP_EVENT.set()
    _stop_all_procs()
    for _ in eval_threads:
        EVAL_QUEUE.put(None)
    for t in gpu_threads + eval_threads:
        t.join(timeout=120)
    STATUS['outcome'] = 'interrupted'
    notify('interrupted', 'Session ' + SESSION + ' was stopped by the user.', priority='high', tags=['stop_sign'])
    raise

heartbeat()
done = [j['job_id'] for j in JOBS if JOB_RESULTS.get(j['job_id'], {}).get('status') == 'completed']
not_completed = [(j['job_id'], JOB_RESULTS.get(j['job_id'], {}).get('status', 'not_started')) for j in JOBS
                 if JOB_RESULTS.get(j['job_id'], {}).get('status') != 'completed']
print()
print('Expected: ' + str(len(JOBS)) + ' | Completed: ' + str(len(done)) + ' | Not completed: ' + str(len(not_completed)))
for jid, status in not_completed:
    print('  ' + status + ': ' + jid)
if not_completed:
    print('Commit a new version to finish them; completed ones on this commit/platform are reused.')
notify('jobs finished', 'Completed ' + str(len(done)) + '/' + str(len(JOBS)) + '. Session ' + SESSION + '.',
       priority='high' if not_completed else 'default',
       tags=['warning'] if not_completed else ['white_check_mark'])

In [ ]:
# 8. Optional: Q4_K_M GGUF on the Kaggle CPU (edge-deployment latency and quantisation check)
#    Runs after the GPU jobs, one at a time, so the CPU timing is not contended.
if not RUN_GGUF_CPU_BENCH:
    print('RUN_GGUF_CPU_BENCH is False; skipping the GGUF benchmark.')
else:
    run_step('build-llama-cpp', 'bash scripts/build_llama_cpp.sh', env={'BUILD_CUDA': '0'})
    os.environ['LLAMA_CPP_DIR'] = str(REPO_DIR / 'vendor' / 'llama.cpp')
    files = hub_files(refresh=True)
    for e in EXP:
        rn = e['run_name']
        jid = rn + '__fine_tuned__gguf_q4_k_m_cpu'
        job = {**e, 'condition': 'fine_tuned', 'backend': 'llamacpp', 'job_id': jid,
               'prompt_schema_mode': e['schema_mode'], 'train_schema_mode': e['schema_mode']}
        if not _time_left_for_job():
            print('[' + jid + '] Not started: too little session time left.')
            continue
        try:
            reused = _reuse_from_hub(job, EVAL_DIR / 'jobs' / jid)
        except Exception:
            reused = None
        if reused is not None:
            print('[' + jid + '] Reusing completed outputs from the Hub.')
            _set_result(jid, reused, 'completed')
            continue
        ggufs = sorted(f for f in files if f.startswith('gguf/' + rn + '/') and f.endswith('-q4_k_m.gguf'))
        if not ggufs:
            print('[' + rn + '] No Q4_K_M GGUF on the Hub; skipping.')
            continue
        base = ggufs[-1].split('/')[-1]
        sidecars = sorted(f for f in files if f.endswith('/' + base + '.json'))
        merge_meta = {}
        if sidecars:
            try:
                sidecar = json.loads(hub_download_file(sidecars[-1]).read_text(encoding='utf-8'))
                merge_meta = sidecar.get('merge_metadata') or {}
            except Exception as exc:
                print('[' + rn + '] Could not read the GGUF sidecar: ' + str(exc))
        # Only Qwen was affected by the wrapper-class bug (history/027.md). Its GGUF must
        # come from the fixed merge, which records model_class in merge_metadata.json.
        if e['model'] == 'qwen3.5-2b' and 'model_class' not in merge_meta:
            print('[' + rn + '] Skipping: no post-fix merge metadata, so this GGUF may not contain the adapter.')
            continue
        print('[' + rn + '] Downloading ' + ggufs[-1] + ' ...', flush=True)
        job['gguf_local'] = str(hub_download_file(ggufs[-1]))
        job['gguf_repo_path'] = ggufs[-1]
        job['gguf_merge_metadata'] = merge_meta
        try:
            ctx = _predict(job, GPU_IDS[0], 'gguf', echo=True)
            rec = _evaluate(job, ctx, 'gguf', echo=True)
            rec['concurrent_gpu_workers'] = 0
            _set_result(jid, rec, 'completed')
        except KeyboardInterrupt:
            raise
        except Exception as exc:
            _set_result(jid, _job_record(job, status='failed', error=str(exc)), 'failed')
            print('[' + jid + '] FAILED (best-effort): ' + str(exc), flush=True)

In [ ]:
# 9. Training telemetry: loss curves from the newest checkpoint's trainer_state.json
#    log_history on the Hub (survives resumes and killed sessions); metrics.jsonl only
#    fills steps after the last checkpoint and supplies VRAM / wall time.
import re
import numpy as np
import pandas as pd
from IPython.display import display
from safetensors import safe_open


def _num(df, col):
    if df is None or df.empty or col not in df.columns:
        return pd.Series(dtype=float)
    return pd.to_numeric(df[col], errors='coerce').dropna()


def _adapter_modules(adir):
    """Counts adapted modules from the saved tensors; fixes the Unsloth leaf-name report."""
    st = Path(adir) / 'adapter_model.safetensors'
    if not st.exists():
        return None, None
    with safe_open(str(st), framework='pt') as f:
        mods = {k.split('.lora_A')[0] for k in f.keys() if '.lora_A' in k}
    return len(mods), sorted({m.split('.')[-1] for m in mods})


files = hub_files(refresh=True)
CKPT_STATE_RE = re.compile(r'^runs/([^/]+)/checkpoint-(\d+)/trainer_state\.json$')
curve_rows, jsonl_rows, state_steps = [], [], {}
for e in EXP:
    rn, key = e['run_name'], e['model'] + '-' + e['variant']
    steps = sorted(int(m.group(2)) for f in files for m in [CKPT_STATE_RE.match(f)] if m and m.group(1) == rn)
    if steps:
        state_steps[rn] = steps[-1]
        state = json.loads(hub_download_file('runs/' + rn + '/checkpoint-' + str(steps[-1])
                                             + '/trainer_state.json').read_text(encoding='utf-8'))
        for entry in state.get('log_history') or []:
            curve_rows.append({**entry, 'run_name': rn, 'model': e['model'], 'variant': e['variant'],
                               'session': None, 'source': 'trainer_state'})
    paths = sorted(f for f in files if f.endswith('/metrics.jsonl')
                   and (f.startswith('results/' + key + '/') or f.startswith('runs/' + rn + '/telemetry/')))
    print(rn + ': trainer_state at step ' + str(state_steps.get(rn, '-')) + ', ' + str(len(paths))
          + ' metrics.jsonl file(s)')
    for path in paths:
        session = path.split('/')[-2]
        try:
            text = hub_download_file(path).read_text(encoding='utf-8')
        except Exception as exc:
            print('  could not read ' + path + ': ' + str(exc))
            continue
        for line in text.splitlines():
            if line.strip():
                row = json.loads(line)
                row.update(run_name=rn, model=e['model'], variant=e['variant'], session=session,
                           source='metrics_jsonl')
                jsonl_rows.append(row)

RAW_CURVES = pd.DataFrame(jsonl_rows)
CURVES = pd.DataFrame(curve_rows + jsonl_rows)
if not CURVES.empty:
    CURVES['kind'] = 'other'
    for col, kind in (('loss', 'train'), ('eval_loss', 'eval'), ('train_runtime', 'summary')):
        if col in CURVES.columns:
            CURVES.loc[CURVES[col].notna(), 'kind'] = kind
    wall = pd.to_numeric(CURVES['wall_time'], errors='coerce') if 'wall_time' in CURVES.columns \
        else pd.Series(np.nan, index=CURVES.index)
    # trainer_state wins; among metrics.jsonl rows the latest session wins.
    CURVES['_order'] = np.where(CURVES['source'] == 'trainer_state', -np.inf, -wall.fillna(0))
    CURVES = (CURVES.sort_values(['run_name', 'kind', 'step', '_order'])
              .drop_duplicates(['run_name', 'kind', 'step'], keep='first')
              .drop(columns='_order').sort_values(['run_name', 'step']).reset_index(drop=True))

train_rows = []
for e in EXP:
    rn = e['run_name']
    tm = e['train_meta'] or {}
    c = CURVES[CURVES['run_name'] == rn] if not CURVES.empty else pd.DataFrame()
    raw = RAW_CURVES[RAW_CURVES['run_name'] == rn] if not RAW_CURVES.empty else pd.DataFrame()
    tr = c[c['kind'] == 'train'] if not c.empty else c
    ev = c[c['kind'] == 'eval'] if not c.empty else c
    hours, n_sessions = float('nan'), 0
    if not raw.empty and 'wall_time' in raw.columns:
        spans = raw.groupby('session')['wall_time'].agg(['min', 'max'])
        hours = float((spans['max'] - spans['min']).sum()) / 3600
        n_sessions = len(spans)
    tr_loss, ev_loss = _num(tr, 'loss'), _num(ev, 'eval_loss')
    logging_steps = ((tm.get('config') or {}).get('train') or {}).get('logging_steps') or 10
    expected_logs = (tm.get('global_step') or 0) // logging_steps
    n_mod, leaves = _adapter_modules(e['adapter_dir'])
    metrics = tm.get('metrics') or {}
    train_rows.append({
        'run_name': rn, 'model': e['model'], 'variant': e['variant'], 'schema_mode': e['schema_mode'],
        'adapter_available': e['adapter_ok'],
        'global_step': tm.get('global_step'), 'epoch': tm.get('epoch'),
        'n_train': tm.get('n_train'), 'n_val': tm.get('n_val'),
        'trainable_parameters': tm.get('trainable_parameters'),
        'adapted_modules': n_mod, 'adapted_leaf_names': '; '.join(leaves or []),
        'target_modules_reported': json.dumps(tm.get('target_modules'), default=str),
        'dtype': tm.get('dtype'), 'device': tm.get('device'),
        'target_only_loss': tm.get('target_only_loss'), 'unsloth': tm.get('unsloth'),
        'curve_source_step': state_steps.get(rn),
        'train_loss_points': int(len(tr_loss)),
        'train_loss_coverage': round(len(tr_loss) / expected_logs, 3) if expected_logs else None,
        'train_sessions_logged': n_sessions,
        'logged_training_hours': round(hours, 2) if hours == hours else None,
        'first_train_loss': float(tr_loss.iloc[0]) if len(tr_loss) else None,
        'final_train_loss_mean_last5': float(tr_loss.tail(5).mean()) if len(tr_loss) else None,
        'final_eval_loss': float(ev_loss.iloc[-1]) if len(ev_loss) else None,
        'min_eval_loss': float(ev_loss.min()) if len(ev_loss) else None,
        'vram_peak_gb_max': float(_num(c, 'vram_peak_gb').max()) if len(_num(c, 'vram_peak_gb')) else tm.get('vram_peak_gb'),
        'trainer_train_loss_final_session': metrics.get('train_loss'),
        'fingerprint_trained': e['fingerprint_trained'], 'fingerprint_now': e['fingerprint_now'],
        'fingerprint_match': e['fingerprint_match'],
    })
TRAINING = pd.DataFrame(train_rows)
display(TRAINING)

In [ ]:
# 10. Aggregate every completed job into thesis tables (CIs, paired tests, integrity checks)
import itertools
import math
from statistics import NormalDist

import numpy as np
from src.evaluation.metrics import _ngram_counts, tokenize_cypher

Z = NormalDist().inv_cdf(0.5 + CI_LEVEL / 2)
ALPHA = 1 - CI_LEVEL
RNG = np.random.default_rng(STATS_SEED)
KEYS = ['job_id', 'run_name', 'model', 'variant', 'schema_mode', 'condition', 'backend']
COMPLEXITY_ORDER = ['easy', 'medium', 'hard', 'extra_hard']
OUT_DIR = REPO_DIR / 'artifacts' / 'thesis' / EVAL_TAG
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
(OUT_DIR / 'tables').mkdir(parents=True)


def _flat(value):
    if isinstance(value, (list, tuple)):
        return '; '.join(str(v) for v in value)
    if isinstance(value, dict):
        return json.dumps(value, default=str)
    if isinstance(value, Path):
        return str(value)
    return value


def _g(series, key):
    value = series.get(key) if series is not None else None
    try:
        return float(value) if value is not None and pd.notna(value) else float('nan')
    except (TypeError, ValueError):
        return float('nan')


def _error_category(r):
    if not str(r.get('predicted_cypher') or '').strip():
        return 'empty_output'
    status = r.get('execution_status')
    if status is None:
        return 'not_executed'
    if status in ('connection_failed', 'excluded_dead_db', 'gold_failed'):
        return status
    if not r.get('pred_executed_ok'):
        if r.get('syntax_valid') is False:
            return 'invalid_query'
        if r.get('syntax_valid') is True:
            return 'runtime_error'
        return 'execution_error'
    return 'correct' if r.get('execution_match') else 'wrong_result'


def _gleu_counts(ref, hyp):
    rc = _ngram_counts(tokenize_cypher(ref), 1, 4)
    hc = _ngram_counts(tokenize_cypher(hyp), 1, 4)
    return sum((rc & hc).values()), sum(hc.values()), sum(rc.values())


# --------------------------------------------------------------------------- #
# Per-row long table
# --------------------------------------------------------------------------- #
row_frames = []
for jid, rec in JOB_RESULTS.items():
    if rec.get('status') != 'completed':
        continue
    scored = EVAL_DIR / 'jobs' / jid / 'predictions.scored.jsonl'
    if not scored.exists():
        print('WARNING: ' + jid + ' has no scored predictions locally; skipped.')
        continue
    out = []
    for line in scored.read_text(encoding='utf-8').splitlines():
        if not line.strip():
            continue
        r = json.loads(line)
        gold = r.get('gold_cypher') or ''
        pred = r.get('predicted_cypher') or ''
        overlap, hyp_n, ref_n = _gleu_counts(gold, pred)
        out.append({
            'job_id': jid, 'run_name': rec.get('run_name'), 'model': rec.get('model'),
            'variant': rec.get('variant'), 'schema_mode': rec.get('schema_mode'),
            'condition': rec.get('condition'), 'backend': rec.get('backend'),
            'prompt_schema_mode': rec.get('prompt_schema_mode'), 'train_schema_mode': rec.get('train_schema_mode'),
            'instance_id': r.get('instance_id'), 'data_source': r.get('data_source'),
            'database_reference': r.get('database_reference'),
            'has_database': bool(str(r.get('database_reference') or '').strip()),
            'complexity': r.get('complexity'),
            'gold_hops': r.get('gold_hops'), 'prompt_tokens': r.get('prompt_tokens'),
            'schema_chars': r.get('schema_chars'),
            'schema_chars_untrimmed': r.get('schema_chars_untrimmed', r.get('schema_chars')),
            'schema_trimmed': r.get('schema_trimmed'), 'prompt_over_budget': r.get('prompt_over_budget'),
            'dead_database': r.get('dead_database'),
            'gleu': r.get('gleu'),
            'gleu_overlap': overlap, 'gleu_hyp_ngrams': hyp_n, 'gleu_ref_ngrams': ref_n,
            'exact_match': bool(r.get('exact_match')),
            'normalised_exact_match': bool(r.get('normalised_exact_match')),
            'empty_output': not pred.strip(),
            'execution_status': r.get('execution_status'),
            'gold_executed_ok': r.get('gold_executed_ok'),
            'pred_executed_ok': r.get('pred_executed_ok'),
            'execution_match': r.get('execution_match'),
            'syntax_valid': r.get('syntax_valid'),
            'gold_row_count': r.get('gold_row_count'), 'pred_row_count': r.get('pred_row_count'),
            'order_sensitive': r.get('order_sensitive'),
            'error_category': _error_category(r),
            'pred_error': (r.get('pred_error') or '')[:300] or None,
            'gold_error': (r.get('gold_error') or '')[:300] or None,
            'question': r.get('question'), 'gold_cypher': gold, 'predicted_cypher': pred,
            'predicted_raw': r.get('predicted_raw'),
        })
    row_frames.append(pd.DataFrame(out))

if not row_frames:
    raise RuntimeError('No completed jobs to aggregate. Run cell 7 first.')
ROWS = pd.concat(row_frames, ignore_index=True)
ROWS['prompt_tokens'] = pd.to_numeric(ROWS['prompt_tokens'], errors='coerce')
ROWS['schema_chars'] = pd.to_numeric(ROWS['schema_chars'], errors='coerce')
ROWS['schema_chars_untrimmed'] = pd.to_numeric(ROWS['schema_chars_untrimmed'], errors='coerce')
JOBS_DF = pd.DataFrame([{k: _flat(v) for k, v in rec.items()} for rec in JOB_RESULTS.values()])
print('Rows: ' + str(len(ROWS)) + ' across ' + str(ROWS['job_id'].nunique()) + ' jobs')


# --------------------------------------------------------------------------- #
# Metrics with confidence intervals
# --------------------------------------------------------------------------- #
def wilson(k, n):
    if n == 0:
        return float('nan'), float('nan')
    p = k / n
    denom = 1 + Z * Z / n
    centre = (p + Z * Z / (2 * n)) / denom
    half = Z * math.sqrt(p * (1 - p) / n + Z * Z / (4 * n * n)) / denom
    lo = max(0.0, centre - half)
    hi = min(1.0, centre + half)
    # At p = 0 or p = 1 the exact bound equals p, but float rounding can leave it a hair
    # past p (e.g. lo = 1e-17 when p = 0). Keep p inside its own interval.
    return min(lo, p), max(hi, p)


def corpus_gleu_counts(o, h, r):
    return min(o / h, o / r) if h > 0 and r > 0 else 0.0


def bootstrap_corpus_gleu(df):
    n = len(df)
    if n == 0:
        return float('nan'), float('nan')
    o = df['gleu_overlap'].to_numpy(float)
    h = df['gleu_hyp_ngrams'].to_numpy(float)
    r = df['gleu_ref_ngrams'].to_numpy(float)
    idx = RNG.integers(0, n, size=(BOOTSTRAP_SAMPLES, n))
    so, sh, sr = o[idx].sum(axis=1), h[idx].sum(axis=1), r[idx].sum(axis=1)
    with np.errstate(divide='ignore', invalid='ignore'):
        vals = np.nan_to_num(np.minimum(so / sh, so / sr))
    lo, hi = np.quantile(vals, [ALPHA / 2, 1 - ALPHA / 2])
    return float(lo), float(hi)


def compute_metrics(df):
    n = len(df)
    out = {'n': n}
    if n == 0:
        return out
    out['google_bleu_corpus'] = corpus_gleu_counts(df['gleu_overlap'].sum(), df['gleu_hyp_ngrams'].sum(),
                                                   df['gleu_ref_ngrams'].sum())
    out['google_bleu_corpus_ci_low'], out['google_bleu_corpus_ci_high'] = bootstrap_corpus_gleu(df)
    out['google_bleu_mean'] = float(pd.to_numeric(df['gleu'], errors='coerce').mean())
    for name in ('exact_match', 'normalised_exact_match'):
        k = int(df[name].sum())
        out[name] = k / n
        out[name + '_ci_low'], out[name + '_ci_high'] = wilson(k, n)
    out['empty_output_rate'] = float(df['empty_output'].mean())
    NOT_SCORED = ['connection_failed', 'excluded_dead_db']
    out['n_has_database'] = int(df['has_database'].sum())
    out['n_connection_failed'] = int(df['execution_status'].eq('connection_failed').sum())
    out['n_excluded_dead_db'] = int(df['execution_status'].eq('excluded_dead_db').sum())
    trimmed = df['schema_trimmed'].dropna()
    if len(trimmed):
        out['n_schema_trimmed'] = int(trimmed.astype(bool).sum())
        out['schema_trimmed_rate'] = float(trimmed.astype(bool).mean())
    out['n_prompt_over_budget'] = int(df['prompt_over_budget'].eq(True).sum())
    executed = df[df['execution_status'].notna() & ~df['execution_status'].isin(NOT_SCORED)]
    gold_ok = executed[executed['gold_executed_ok'].eq(True)]
    out['n_executed'] = len(executed)
    out['n_gold_ok'] = len(gold_ok)
    out['n_gold_failed'] = len(executed) - len(gold_ok)
    if len(executed):
        out['execution_accuracy_toolkit'] = float(executed['execution_match'].eq(True).mean())
        out['executable_rate'] = float(executed['pred_executed_ok'].eq(True).mean())
    if len(gold_ok):
        k = int(gold_ok['execution_match'].eq(True).sum())
        out['execution_accuracy'] = k / len(gold_ok)
        out['execution_accuracy_ci_low'], out['execution_accuracy_ci_high'] = wilson(k, len(gold_ok))
        out['executable_rate_gold_ok'] = float(gold_ok['pred_executed_ok'].eq(True).mean())
    validated = df[df['syntax_valid'].notna()]
    if len(validated):
        out['syntax_valid_rate'] = float(validated['syntax_valid'].eq(True).mean())
    tok = df['prompt_tokens'].dropna()
    tok = tok[tok >= 0]
    if len(tok):
        out['prompt_tokens_mean'] = float(tok.mean())
        out['prompt_tokens_median'] = float(tok.median())
        out['prompt_tokens_p95'] = float(tok.quantile(0.95))
        out['prompt_tokens_max'] = float(tok.max())
    return out


def table_by(extra):
    recs = []
    for keys, sub in ROWS.groupby(KEYS + extra, dropna=False, sort=False):
        rec = dict(zip(KEYS + extra, keys))
        rec.update(compute_metrics(sub))
        recs.append(rec)
    return pd.DataFrame(recs)


OVERALL = table_by([])
job_cols = [c for c in ('job_id', 'seconds_per_item', 'generation_seconds', 'batch_size', 'max_new_tokens',
                        'gpu', 'gpu_index', 'platform', 'concurrent_gpu_workers', 'git_commit',
                        'adapter_verified') if c in JOBS_DF.columns]
OVERALL = OVERALL.merge(JOBS_DF[job_cols], on='job_id', how='left')
OVERALL = OVERALL.sort_values(['model', 'schema_mode', 'backend', 'condition']).reset_index(drop=True)

COND_LABELS = {'fine_tuned': 'fine-tuned', 'zero_shot': 'zero-shot',
               'cross_schema': 'fine-tuned (trained enhanced, exact-match prompts)'}


def condition_label(model, condition):
    label = COND_LABELS.get(condition, str(condition))
    if condition == 'zero_shot' and PLAIN_TEXT_PROMPT.get(model):
        label += ' (base checkpoint, plain-text prompt)'
    return label


OVERALL['condition_label'] = [condition_label(m, c) for m, c in zip(OVERALL['model'], OVERALL['condition'])]

BY_COMPLEXITY = table_by(['complexity'])
BY_COMPLEXITY['_rank'] = BY_COMPLEXITY['complexity'].map({c: i for i, c in enumerate(COMPLEXITY_ORDER)})
BY_COMPLEXITY = BY_COMPLEXITY.sort_values(['job_id', '_rank']).drop(columns='_rank').reset_index(drop=True)
BY_SOURCE = table_by(['data_source'])
BY_DATABASE = table_by(['database_reference'])
BY_DB_ACCESS = table_by(['has_database'])
BY_COMPLEXITY_DB_ACCESS = table_by(['has_database', 'complexity'])
ERRORS = ROWS.groupby(KEYS + ['error_category']).size().unstack(fill_value=0).reset_index()

# --------------------------------------------------------------------------- #
# Paired comparisons on identical rows
# --------------------------------------------------------------------------- #
JOB_KEYS = ROWS[KEYS].drop_duplicates().set_index('job_id')
MODELS = list(dict.fromkeys(e['model'] for e in EXP))
SCHEMA_FAMILIES = ('schema_filtering', 'schema_inference_effect', 'schema_training_effect')
JUDGED = ['compared', 'pred_failed']


def jid_for(model, variant, condition, backend='hf'):
    m = JOB_KEYS[(JOB_KEYS['model'] == model) & (JOB_KEYS['variant'] == variant)
                 & (JOB_KEYS['condition'] == condition) & (JOB_KEYS['backend'] == backend)]
    return m.index[0] if len(m) else None


def mcnemar_exact(b, c):
    n = b + c
    if n == 0:
        return 1.0
    tail = sum(math.comb(n, i) for i in range(min(b, c) + 1))
    return min(1.0, 2 * tail / 2 ** n)


def paired_diff_ci(a_vals, b_vals):
    """Paired bootstrap of mean(b - a): CI, a (1 - 2*alpha) CI for TOST, and a two-sided p."""
    d = np.asarray(b_vals, float) - np.asarray(a_vals, float)
    idx = RNG.integers(0, len(d), size=(BOOTSTRAP_SAMPLES, len(d)))
    means = d[idx].mean(axis=1)
    lo, hi = np.quantile(means, [ALPHA / 2, 1 - ALPHA / 2])
    lo90, hi90 = np.quantile(means, [ALPHA, 1 - ALPHA])
    return {'difference_b_minus_a': float(d.mean()), 'diff_ci_low': float(lo), 'diff_ci_high': float(hi),
            'tost_ci_low': float(lo90), 'tost_ci_high': float(hi90),
            'equivalent_within_margin': bool(lo90 > -EQUIV_MARGIN and hi90 < EQUIV_MARGIN),
            'bootstrap_p': min(1.0, 2 * min(float((means <= 0).mean()), float((means >= 0).mean())))}


def paired_compare(a, b, label, family):
    A = ROWS[ROWS['job_id'] == a].drop_duplicates('instance_id').set_index('instance_id')
    B = ROWS[ROWS['job_id'] == b].drop_duplicates('instance_id').set_index('instance_id')
    common = A.index.intersection(B.index)
    A, B = A.loc[common], B.loc[common]
    base = {'comparison': label, 'family': family, 'job_a': a, 'job_b': b}
    results = []
    for metric in ('exact_match', 'normalised_exact_match', 'execution_match'):
        if metric == 'execution_match':
            # Only rows judged on both sides: gold ran, and neither side hit an outage or a dead database.
            mask = (A['gold_executed_ok'].eq(True) & B['gold_executed_ok'].eq(True)
                    & A['execution_status'].isin(JUDGED) & B['execution_status'].isin(JUDGED)).to_numpy()
        else:
            mask = np.ones(len(common), dtype=bool)
        n = int(mask.sum())
        if n == 0:
            continue
        a_ok = A[metric].eq(True).to_numpy()[mask]
        b_ok = B[metric].eq(True).to_numpy()[mask]
        a_only, b_only = int((a_ok & ~b_ok).sum()), int((b_ok & ~a_ok).sum())
        stats = paired_diff_ci(a_ok, b_ok)
        stats.pop('bootstrap_p')
        results.append({**base, 'metric': metric, 'n_pairs': n, 'a_value': float(a_ok.mean()),
                        'b_value': float(b_ok.mean()), **stats,
                        'a_only_correct': a_only, 'b_only_correct': b_only,
                        'test': 'exact McNemar (two-sided)', 'p_value': mcnemar_exact(a_only, b_only),
                        'p_value_is_bound': False})
    ga = pd.to_numeric(A['gleu'], errors='coerce').fillna(0).to_numpy(float)
    gb = pd.to_numeric(B['gleu'], errors='coerce').fillna(0).to_numpy(float)
    if len(ga):
        stats = paired_diff_ci(ga, gb)
        p_raw = stats.pop('bootstrap_p')
        floor = 1.0 / BOOTSTRAP_SAMPLES
        results.append({**base, 'metric': 'sentence_gleu_mean', 'n_pairs': len(ga), 'a_value': float(ga.mean()),
                        'b_value': float(gb.mean()), **stats,
                        'test': 'paired bootstrap (percentile)', 'p_value': max(p_raw, floor),
                        'p_value_is_bound': p_raw < floor})
    return results


def holm(pvals):
    p = np.asarray(pvals, float)
    order = np.argsort(p)
    adj = np.empty_like(p)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, min(1.0, (len(p) - rank) * p[i]))
        adj[i] = running
    return adj


COMPARISONS = []
for e in EXP:
    m, v = e['model'], e['variant']
    a, b = jid_for(m, v, 'zero_shot'), jid_for(m, v, 'fine_tuned')
    if a and b:
        COMPARISONS.append((a, b, m + ' ' + v + ': zero-shot vs fine-tuned', 'fine_tuning_effect'))
for m in MODELS:
    for cond in ('fine_tuned', 'zero_shot'):
        a, b = jid_for(m, 'enhanced', cond), jid_for(m, 'pruned', cond)
        if a and b:
            COMPARISONS.append((a, b, m + ' ' + cond + ': enhanced vs exact-match schema', 'schema_filtering'))
ft_enhanced = [(m, jid_for(m, 'enhanced', 'fine_tuned')) for m in MODELS]
ft_enhanced = [x for x in ft_enhanced if x[1]]
for (ma, a), (mb, b) in itertools.combinations(ft_enhanced, 2):
    COMPARISONS.append((a, b, 'fine-tuned enhanced: ' + ma + ' vs ' + mb, 'model_comparison'))
for e in EXP:
    a = jid_for(e['model'], e['variant'], 'fine_tuned', 'hf')
    b = jid_for(e['model'], e['variant'], 'fine_tuned', 'llamacpp')
    if a and b:
        COMPARISONS.append((a, b, e['run_name'] + ': 4-bit HF (GPU) vs Q4_K_M GGUF (CPU)', 'quantisation'))
for m in MODELS:
    ee = jid_for(m, 'enhanced', 'fine_tuned')
    ex = jid_for(m, 'enhanced', 'cross_schema')
    pp = jid_for(m, 'pruned', 'fine_tuned')
    if ee and ex:
        COMPARISONS.append((ee, ex, m + ': enhanced-trained, enhanced vs exact-match prompts', 'schema_inference_effect'))
    if ex and pp:
        COMPARISONS.append((ex, pp, m + ': exact-match prompts, enhanced-trained vs exact-match-trained',
                            'schema_training_effect'))

paired_rows = []
for a, b, label, family in COMPARISONS:
    paired_rows.extend(paired_compare(a, b, label, family))
PAIRED = pd.DataFrame(paired_rows)
if not PAIRED.empty:
    PAIRED['p_holm'] = PAIRED.groupby(['family', 'metric'])['p_value'].transform(lambda s: holm(s.to_numpy()))
    PAIRED['significant_holm'] = PAIRED['p_holm'] < ALPHA
    floor_txt = '< ' + format(1.0 / BOOTSTRAP_SAMPLES, '.2g')
    PAIRED['p_value_display'] = [floor_txt if bound else format(p, '.3g')
                                 for p, bound in zip(PAIRED['p_value'], PAIRED['p_value_is_bound'])]
    non_schema = ~PAIRED['family'].isin(SCHEMA_FAMILIES)
    PAIRED.loc[non_schema, ['tost_ci_low', 'tost_ci_high']] = np.nan
    PAIRED['equivalent_within_margin'] = PAIRED['equivalent_within_margin'].astype(object)
    PAIRED.loc[non_schema, 'equivalent_within_margin'] = None

# --------------------------------------------------------------------------- #
# Schema filtering: token efficiency vs accuracy (sub-question 1)
# --------------------------------------------------------------------------- #
OV = OVERALL.set_index('job_id')
sf_rows = []
for m in MODELS:
    for cond in ('fine_tuned', 'zero_shot'):
        a, b = jid_for(m, 'enhanced', cond), jid_for(m, 'pruned', cond)
        if not (a and b):
            continue
        oa, ob = OV.loc[a], OV.loc[b]
        A = ROWS[ROWS['job_id'] == a].drop_duplicates('instance_id').set_index('instance_id')
        B = ROWS[ROWS['job_id'] == b].drop_duplicates('instance_id').set_index('instance_id')
        common = A.index.intersection(B.index)
        # Filter effect only: untrimmed schema sizes (training-budget trimming excluded).
        ratio = (B.loc[common, 'schema_chars_untrimmed']
                 / A.loc[common, 'schema_chars_untrimmed'].replace(0, np.nan)).astype(float)
        p_ex = PAIRED[(PAIRED['job_a'] == a) & (PAIRED['job_b'] == b)
                      & (PAIRED['metric'] == 'execution_match')] if not PAIRED.empty else PAIRED
        ex = p_ex.iloc[0] if len(p_ex) else None
        pruned_ids = ratio.index[(ratio < PRUNED_RATIO_THRESHOLD).to_numpy()]
        judged_a = A.loc[pruned_ids, 'execution_status'].isin(JUDGED)
        judged_b = B.loc[pruned_ids, 'execution_status'].isin(JUDGED)
        both = (judged_a & judged_b).to_numpy()
        ids = pruned_ids[both]
        a_ok = A.loc[ids, 'execution_match'].eq(True).to_numpy()
        b_ok = B.loc[ids, 'execution_match'].eq(True).to_numpy()
        pr = paired_diff_ci(a_ok, b_ok) if len(ids) else {}
        tok_a, tok_b = _g(oa, 'prompt_tokens_mean'), _g(ob, 'prompt_tokens_mean')
        nan = float('nan')
        sf_rows.append({
            'model': m, 'condition': cond, 'job_enhanced': a, 'job_exact_match': b,
            'prompt_tokens_mean_enhanced': tok_a, 'prompt_tokens_mean_exact_match': tok_b,
            'prompt_token_reduction_pct': 100 * (1 - tok_b / tok_a) if tok_a else nan,
            'prompt_tokens_p95_enhanced': _g(oa, 'prompt_tokens_p95'),
            'prompt_tokens_p95_exact_match': _g(ob, 'prompt_tokens_p95'),
            'n_schema_trimmed_enhanced': _g(oa, 'n_schema_trimmed'),
            'n_schema_trimmed_exact_match': _g(ob, 'n_schema_trimmed'),
            'schema_chars_median_ratio': float(ratio.median()),
            'share_rows_le5pct_schema_reduction': float((ratio >= PRUNED_RATIO_THRESHOLD).mean()),
            'execution_accuracy_enhanced': _g(oa, 'execution_accuracy'),
            'execution_accuracy_exact_match': _g(ob, 'execution_accuracy'),
            'paired_ea_delta': float(ex['difference_b_minus_a']) if ex is not None else nan,
            'paired_ea_delta_ci_low': float(ex['diff_ci_low']) if ex is not None else nan,
            'paired_ea_delta_ci_high': float(ex['diff_ci_high']) if ex is not None else nan,
            'paired_ea_n': int(ex['n_pairs']) if ex is not None else 0,
            'execution_match_p_value': float(ex['p_value']) if ex is not None else nan,
            'tost_margin': EQUIV_MARGIN,
            'tost_ci_low': float(ex['tost_ci_low']) if ex is not None else nan,
            'tost_ci_high': float(ex['tost_ci_high']) if ex is not None else nan,
            'equivalent_within_margin': ex['equivalent_within_margin'] if ex is not None else None,
            'pruned_rows_n': int(len(pruned_ids)), 'pruned_rows_n_judged': int(len(ids)),
            'pruned_rows_ea_enhanced': float(a_ok.mean()) if len(ids) else nan,
            'pruned_rows_ea_exact_match': float(b_ok.mean()) if len(ids) else nan,
            'pruned_rows_ea_delta': pr.get('difference_b_minus_a', nan),
            'pruned_rows_delta_ci_low': pr.get('diff_ci_low', nan),
            'pruned_rows_delta_ci_high': pr.get('diff_ci_high', nan),
            'pruned_rows_mcnemar_p': (mcnemar_exact(int((a_ok & ~b_ok).sum()), int((b_ok & ~a_ok).sum()))
                                      if len(ids) else nan),
            'google_bleu_enhanced': _g(oa, 'google_bleu_corpus'),
            'google_bleu_exact_match': _g(ob, 'google_bleu_corpus'),
            'exact_match_enhanced': _g(oa, 'exact_match'), 'exact_match_exact_match': _g(ob, 'exact_match'),
            'seconds_per_item_enhanced': _g(oa, 'seconds_per_item'),
            'seconds_per_item_exact_match': _g(ob, 'seconds_per_item'),
        })
SCHEMA_FILTER = pd.DataFrame(sf_rows)

# --------------------------------------------------------------------------- #
# Integrity checks
# --------------------------------------------------------------------------- #
INTEGRITY = []
hf_jobs = sorted(ROWS.loc[ROWS['backend'] == 'hf', 'job_id'].unique())
id_sets = {j: frozenset(ROWS.loc[ROWS['job_id'] == j, 'instance_id']) for j in hf_jobs}
INTEGRITY.append({'check': 'identical evaluation rows across all GPU jobs', 'subject': 'all',
                  'passed': len(set(id_sets.values())) <= 1,
                  'detail': ', '.join(j + '=' + str(len(s)) for j, s in id_sets.items())})
for jid, rec in JOB_RESULTS.items():
    if rec.get('status') == 'completed' and rec.get('condition') in ADAPTER_CONDITIONS and rec.get('backend') == 'hf':
        ver = rec.get('adapter_verified') or []
        INTEGRITY.append({'check': 'adapter tensors verified at load (history/027.md)', 'subject': jid,
                          'passed': bool(ver), 'detail': _flat(ver)})
for a, b, label, family in COMPARISONS:
    if family != 'fine_tuning_effect':
        continue
    A = ROWS[ROWS['job_id'] == a].drop_duplicates('instance_id').set_index('instance_id')['predicted_cypher']
    B = ROWS[ROWS['job_id'] == b].drop_duplicates('instance_id').set_index('instance_id')['predicted_cypher']
    common = A.index.intersection(B.index)
    identical = float((A.loc[common] == B.loc[common]).mean()) if len(common) else float('nan')
    INTEGRITY.append({'check': 'fine-tuned outputs differ from zero-shot outputs', 'subject': b,
                      'passed': identical < 0.9, 'detail': format(identical, '.1%') + ' of predictions identical to ' + a})
for _, row in OVERALL.iterrows():
    n_exec = _g(row, 'n_executed')
    if n_exec and n_exec == n_exec:
        share = _g(row, 'n_gold_failed') / n_exec
        INTEGRITY.append({'check': 'gold queries execute on the demo server (<5% failures)', 'subject': row['job_id'],
                          'passed': share < 0.05, 'detail': format(share, '.1%') + ' of executed rows had a failing gold query'})
for e in EXP:
    INTEGRITY.append({'check': 'training config fingerprint matches current config', 'subject': e['run_name'],
                      'passed': e['fingerprint_match'],
                      'detail': str(e['fingerprint_trained']) + ' vs ' + str(e['fingerprint_now'])})
for job in JOBS:
    rec = JOB_RESULTS.get(job['job_id'], {})
    status = rec.get('status', 'not_started')
    INTEGRITY.append({'check': 'expected job completed', 'subject': job['job_id'],
                      'passed': status == 'completed',
                      'detail': status + ((': ' + str(rec.get('error'))) if rec.get('error') else '')})
completed_recs = [r for r in JOB_RESULTS.values() if r.get('status') == 'completed']
BUNDLE_COMMITS = sorted({str(r.get('git_commit')) for r in completed_recs})
BUNDLE_PLATFORMS = sorted({r.get('platform') or 'colab' for r in completed_recs})
INTEGRITY.append({'check': 'all jobs from one code commit', 'subject': 'all',
                  'passed': len(BUNDLE_COMMITS) <= 1, 'detail': ', '.join(c[:10] for c in BUNDLE_COMMITS)})
INTEGRITY.append({'check': 'all jobs from one platform', 'subject': 'all',
                  'passed': len(BUNDLE_PLATFORMS) <= 1, 'detail': ', '.join(BUNDLE_PLATFORMS)})
INTEGRITY.append({'check': 'code version pinned with GIT_REF', 'subject': 'all',
                  'passed': bool(GIT_REF), 'detail': GIT_REF or '(blank)'})
for _, row in OVERALL.iterrows():
    ncf = _g(row, 'n_connection_failed')
    if ncf == ncf:
        INTEGRITY.append({'check': 'no Neo4j connection failures', 'subject': row['job_id'],
                          'passed': ncf == 0, 'detail': format(ncf, '.0f') + ' rows connection_failed'})

gold_rows = (ROWS[ROWS['execution_status'].notna() & ROWS['execution_status'].ne('connection_failed')]
             .drop_duplicates(['database_reference', 'instance_id']))
if len(gold_rows):
    GOLD_HEALTH = (gold_rows.groupby('database_reference')
                   .agg(n=('instance_id', 'size'),
                        n_gold_failed=('gold_executed_ok', lambda s: int((~s.eq(True)).sum())),
                        sample_gold_error=('gold_error', lambda s: next((x for x in s if x), None)))
                   .reset_index())
    GOLD_HEALTH['gold_failure_rate'] = GOLD_HEALTH['n_gold_failed'] / GOLD_HEALTH['n']
    GOLD_HEALTH = GOLD_HEALTH[GOLD_HEALTH['n_gold_failed'] > 0].sort_values('gold_failure_rate', ascending=False)
else:
    GOLD_HEALTH = pd.DataFrame(columns=['database_reference', 'n', 'n_gold_failed', 'sample_gold_error',
                                        'gold_failure_rate'])
DEAD_DBS = GOLD_HEALTH.loc[GOLD_HEALTH['gold_failure_rate'].eq(1.0), 'database_reference'].tolist()
INTEGRITY.append({'check': 'databases where every gold query fails (outside execution_accuracy by construction)',
                  'subject': 'all', 'passed': None, 'detail': ', '.join(DEAD_DBS) or 'none'})
if DEAD_DBS:
    print('Dead databases: ' + ', '.join(DEAD_DBS))
    display(GOLD_HEALTH)
INTEGRITY_DF = pd.DataFrame(INTEGRITY)


# --------------------------------------------------------------------------- #
# Headline table and published baselines
# --------------------------------------------------------------------------- #
def _ci(row, metric):
    v = _g(row, metric)
    if v != v:
        return ''
    lo, hi = _g(row, metric + '_ci_low'), _g(row, metric + '_ci_high')
    if lo != lo:
        return format(v, '.3f')
    return format(v, '.3f') + ' [' + format(lo, '.3f') + ', ' + format(hi, '.3f') + ']'


def _fmt(v, spec='.3f'):
    return '' if v is None or pd.isna(v) else format(float(v), spec)


HEADLINE = pd.DataFrame([{
    'Model': r['model'], 'Schema': r['schema_mode'], 'Condition': r['condition_label'],
    'Backend': r['backend'], 'N': int(r['n']),
    'GLEU': _ci(r, 'google_bleu_corpus'), 'EM': _ci(r, 'exact_match'),
    'Norm. EM': _fmt(r.get('normalised_exact_match')),
    'Exec. acc.': _ci(r, 'execution_accuracy'), 'Exec. N': _fmt(r.get('n_gold_ok'), '.0f'),
    'Executable': _fmt(r.get('executable_rate_gold_ok')),
    'Trimmed': _fmt(r.get('n_schema_trimmed'), '.0f'),
    'Prompt tokens': _fmt(r.get('prompt_tokens_mean'), '.0f'),
    's/item': _fmt(r.get('seconds_per_item'), '.2f'), 'Batch': _fmt(r.get('batch_size'), '.0f'),
    'Platform': r.get('platform') if isinstance(r.get('platform'), str) else 'colab',
    'Commit': str(r.get('git_commit') or '')[:7],
} for _, r in OVERALL.iterrows()])

PUBLISHED = (pd.DataFrame(PUBLISHED_BASELINES) if PUBLISHED_BASELINES
             else pd.DataFrame(columns=['source', 'model', 'metric', 'value', 'notes']))

print('Headline (' + str(int(CI_LEVEL * 100)) + '% CI in brackets):')
display(HEADLINE)
if not PAIRED.empty:
    print('Paired tests:')
    display(PAIRED[['comparison', 'metric', 'n_pairs', 'a_value', 'b_value', 'difference_b_minus_a',
                    'p_value_display', 'p_holm', 'significant_holm']])
if not SCHEMA_FILTER.empty:
    print('Schema filtering:')
    display(SCHEMA_FILTER)
failed_checks = INTEGRITY_DF[INTEGRITY_DF['passed'].eq(False)]
print('Integrity checks: ' + str(len(INTEGRITY_DF)) + ' run, ' + str(len(failed_checks)) + ' failed')
if len(failed_checks):
    display(failed_checks)

In [ ]:
# 11. Figures for the thesis (PNG at 300 dpi + vector PDF)
import matplotlib.pyplot as plt

FIG_DIR = OUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 300, 'font.size': 9})


def save_fig(fig, name):
    for ext in ('png', 'pdf'):
        fig.savefig(FIG_DIR / (name + '.' + ext), bbox_inches='tight')
    plt.show()
    plt.close(fig)


# Figures 1-3: headline metrics, zero-shot vs fine-tuned, with CIs.
hf = OVERALL[(OVERALL['backend'] == 'hf') & OVERALL['condition'].isin(['zero_shot', 'fine_tuned'])].copy()
hf['label'] = hf['model'] + LF + hf['schema_mode']
labels = list(dict.fromkeys(hf['label']))
x = np.arange(len(labels))
width = 0.38
for metric, title, name in (('execution_accuracy', 'Execution accuracy', 'fig1_execution_accuracy'),
                            ('google_bleu_corpus', 'Google-BLEU (corpus)', 'fig2_google_bleu'),
                            ('exact_match', 'Exact match', 'fig3_exact_match')):
    if metric not in hf.columns or hf[metric].isna().all():
        continue
    fig, ax = plt.subplots(figsize=(max(6, 1.6 * len(labels)), 4))
    for k, cond in enumerate(('zero_shot', 'fine_tuned')):
        sub = hf[hf['condition'] == cond].set_index('label').reindex(labels)
        if sub[metric].isna().all():
            continue
        vals = sub[metric].to_numpy(float)
        lo = sub[metric + '_ci_low'].to_numpy(float) if metric + '_ci_low' in sub.columns else vals
        hi = sub[metric + '_ci_high'].to_numpy(float) if metric + '_ci_high' in sub.columns else vals
        # Matplotlib rejects negative error bars. A percentile bootstrap CI (corpus GLEU)
        # need not contain the point estimate, so clip the bar at the estimate.
        yerr = np.clip(np.nan_to_num(np.vstack([vals - lo, hi - vals])), 0, None)
        ax.bar(x + (k - 0.5) * width, np.nan_to_num(vals), width, yerr=yerr, capsize=3,
               label=cond.replace('_', '-'))
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylim(0, 1)
    ax.set_ylabel(title)
    ax.set_title(title + ' (' + SPLIT + ' split, n=' + LIMIT_LABEL + ', ' + str(int(CI_LEVEL * 100)) + '% CI)')
    ax.grid(axis='y', alpha=0.3)
    ax.legend()
    save_fig(fig, name)

# Figure 4: execution accuracy by query complexity (fine-tuned models).
ft = BY_COMPLEXITY[(BY_COMPLEXITY['condition'] == 'fine_tuned') & (BY_COMPLEXITY['backend'] == 'hf')]
if not ft.empty and 'execution_accuracy' in ft.columns and ft['execution_accuracy'].notna().any():
    pv = ft.pivot_table(index='complexity', columns='run_name', values='execution_accuracy',
                        aggfunc='first').reindex(COMPLEXITY_ORDER)
    counts = ft.groupby('complexity')['n_gold_ok'].max().reindex(COMPLEXITY_ORDER) if 'n_gold_ok' in ft.columns else None
    fig, ax = plt.subplots(figsize=(8, 4))
    pv.plot(kind='bar', ax=ax, rot=0, width=0.8)
    ticks = []
    for c in COMPLEXITY_ORDER:
        v = counts.get(c) if counts is not None else None
        ticks.append(c + LF + 'n=' + (str(int(v)) if v is not None and pd.notna(v) else '0'))
    ax.set_xticklabels(ticks)
    ax.set_ylim(0, 1)
    ax.set_ylabel('Execution accuracy')
    ax.set_xlabel('Query complexity (gold query)')
    ax.set_title('Fine-tuned execution accuracy by query complexity')
    ax.grid(axis='y', alpha=0.3)
    ax.legend(fontsize=7)
    save_fig(fig, 'fig4_execution_accuracy_by_complexity')

# Figure 5: training and validation loss curves.
if not CURVES.empty and 'loss' in CURVES.columns:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for rn, g in CURVES.groupby('run_name'):
        tr = g[g['kind'] == 'train']
        if not tr.empty:
            axes[0].plot(tr['step'], tr['loss'], label=rn, linewidth=1)
        if 'eval_loss' in g.columns:
            ev = g[g['kind'] == 'eval']
            if not ev.empty:
                axes[1].plot(ev['step'], ev['eval_loss'], marker='o', markersize=3, label=rn, linewidth=1)
    axes[0].set_title('Training loss')
    axes[1].set_title('Validation loss')
    for ax in axes:
        ax.set_xlabel('Optimizer step')
        ax.grid(alpha=0.3)
        ax.legend(fontsize=7)
    save_fig(fig, 'fig5_training_curves')

# Figure 6: prompt length under each schema mode.
tok = ROWS[ROWS['backend'] == 'hf'].drop_duplicates(['model', 'schema_mode', 'instance_id'])
groups = [(m + LF + s, g['prompt_tokens'].dropna().to_numpy()) for (m, s), g in tok.groupby(['model', 'schema_mode'])]
groups = [gr for gr in groups if len(gr[1])]
if groups:
    fig, ax = plt.subplots(figsize=(max(6, 1.4 * len(groups)), 4))
    ax.boxplot([g for _, g in groups], showfliers=False)
    ax.set_xticks(range(1, len(groups) + 1))
    ax.set_xticklabels([lbl for lbl, _ in groups])
    ax.set_ylabel('Prompt tokens')
    ax.set_title('Prompt length by schema mode (outliers hidden)')
    ax.grid(axis='y', alpha=0.3)
    save_fig(fig, 'fig6_prompt_tokens')

# Figure 7: error categories per job.
cats = [c for c in ('correct', 'wrong_result', 'invalid_query', 'runtime_error', 'execution_error',
                    'empty_output', 'gold_failed', 'excluded_dead_db', 'connection_failed', 'not_executed')
        if c in ERRORS.columns]
if cats:
    share = ERRORS.set_index('job_id')[cats]
    share = share.div(share.sum(axis=1), axis=0)
    fig, ax = plt.subplots(figsize=(9, 0.45 * len(share) + 1.5))
    share.plot(kind='barh', stacked=True, ax=ax, width=0.8)
    ax.set_xlim(0, 1)
    ax.set_xlabel('Share of rows')
    ax.set_title('Outcome categories per job')
    ax.legend(fontsize=7, bbox_to_anchor=(1.01, 1), loc='upper left')
    save_fig(fig, 'fig7_error_categories')

print('Figures saved to ' + str(FIG_DIR))

In [ ]:
# 12. Write the structured results bundle, upload it, save it with the version and finish
import importlib.metadata as importlib_metadata


def _records(df):
    if df is None or df.empty:
        return []
    return json.loads(df.to_json(orient='records', date_format='iso', default_handler=str))


def _version(dist):
    try:
        return importlib_metadata.version(dist)
    except Exception:
        return None


created = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')
MANIFEST = {
    'title': 'Text2Cypher LoRA fine-tuning: final evaluation results',
    'created_utc': created, 'session': SESSION, 'eval_tag': EVAL_TAG, 'split': SPLIT,
    'eval_limit': EVAL_LIMIT, 'sample_seed': SAMPLE_SEED, 'gguf_bench_limit': GGUF_BENCH_LIMIT,
    'git_repo': GITHUB_REPO, 'git_ref': GIT_REF, 'git_commit': GIT_COMMIT, 'git_dirty': GIT_DIRTY,
    'hub_repo': HUB_REPO, 'hub_eval_prefix': HUB_EVAL_PREFIX,
    'platform_name': PLATFORM, 'kaggle_run_type': RUN_TYPE,
    'gpu': GPU_LABEL, 'gpus': GPUS, 'gpu_workers': GPU_IDS, 'eval_workers': N_EVAL_WORKERS,
    'python': platform.python_version(), 'platform': platform.platform(),
    'packages': {d: _version(d) for d in ('torch', 'transformers', 'peft', 'bitsandbytes', 'accelerate',
                                          'huggingface_hub', 'neo4j', 'pandas', 'numpy')},
    'decoding': 'greedy (temperature 0.0); max_new_tokens and batch size from each config',
    'execution_provider': 'Neo4j Labs demo server (neo4j+s://demo.neo4jlabs.com:7687)',
    'latency_note': ('s/item is NOT comparable across platforms (Colab: one T4 alone; Kaggle: '
                     + str(len(GPU_IDS)) + ' concurrent predictions on 4 vCPUs) or across batch sizes '
                     '(Gemma/Granite 4, Qwen 8). Compare latency only within one platform and batch size.'),
    'prompt_budget': 'max_seq_length minus generation.prompt_target_reserve (default max_new_tokens); '
                     'schema property lines trimmed as in training',
    'dead_db_policy': DEAD_DB_POLICY, 'dead_databases': DEAD_DBS,
    'equivalence_margin': EQUIV_MARGIN, 'pruned_ratio_threshold': PRUNED_RATIO_THRESHOLD,
    'statistics': {
        'ci_level': CI_LEVEL, 'bootstrap_samples': BOOTSTRAP_SAMPLES, 'stats_seed': STATS_SEED,
        'proportion_ci': 'Wilson score interval',
        'gleu_ci': 'percentile bootstrap over rows',
        'paired_tests': 'exact McNemar for binary metrics; paired bootstrap for mean sentence GLEU; '
                        'Holm correction within each family x metric',
        'equivalence': 'TOST via the (1 - 2 alpha) paired bootstrap CI, schema families only',
    },
    'experiments': [{k: _flat(e.get(k)) for k in ('model', 'variant', 'config', 'run_name', 'schema_mode',
                                                  'adapter_ok', 'fingerprint_trained', 'fingerprint_now',
                                                  'fingerprint_match')} for e in EXP],
    'jobs': {jid: rec.get('status') for jid, rec in JOB_RESULTS.items()},
    'jobs_expected': [j['job_id'] for j in JOBS],
    'jobs_not_run': [j['job_id'] for j in JOBS if JOB_RESULTS.get(j['job_id'], {}).get('status') != 'completed'],
    'jobs_skipped_for_time': SKIPPED_FOR_TIME,
    'job_platforms': {jid: rec.get('platform') or 'colab' for jid, rec in JOB_RESULTS.items()},
    'job_commits': {jid: rec.get('git_commit') for jid, rec in JOB_RESULTS.items()},
    'mixed_commits': len(BUNDLE_COMMITS) > 1, 'mixed_platforms': len(BUNDLE_PLATFORMS) > 1,
}

TABLES = {
    'headline': HEADLINE, 'overall_metrics': OVERALL, 'metrics_by_complexity': BY_COMPLEXITY,
    'metrics_by_data_source': BY_SOURCE, 'metrics_by_database': BY_DATABASE,
    'metrics_by_database_access': BY_DB_ACCESS,
    'metrics_by_complexity_and_database_access': BY_COMPLEXITY_DB_ACCESS,
    'gold_query_health': GOLD_HEALTH,
    'error_categories': ERRORS, 'paired_tests': PAIRED, 'schema_filtering': SCHEMA_FILTER,
    'training_summary': TRAINING, 'training_curves': CURVES, 'integrity_checks': INTEGRITY_DF,
    'jobs': JOBS_DF, 'published_baselines': PUBLISHED,
}
for name, df in TABLES.items():
    if df is not None:
        df.to_csv(OUT_DIR / 'tables' / (name + '.csv'), index=False)
try:
    ROWS.to_parquet(OUT_DIR / 'predictions_long.parquet', index=False)
except Exception as exc:
    print('Parquet export failed (' + str(exc) + '); the gzipped CSV is still written.')
ROWS.drop(columns=['predicted_raw']).to_csv(OUT_DIR / 'predictions_long.csv.gz', index=False, compression='gzip')
try:
    HEADLINE.to_latex(OUT_DIR / 'tables' / 'headline.tex', index=False)
except Exception as exc:
    print('LaTeX export skipped: ' + str(exc))

results = {'manifest': MANIFEST}
results.update({name: _records(df) for name, df in TABLES.items() if name != 'training_curves'})
(OUT_DIR / 'results.json').write_text(json.dumps(results, indent=2, default=str), encoding='utf-8')
(OUT_DIR / 'manifest.json').write_text(json.dumps(MANIFEST, indent=2, default=str), encoding='utf-8')

jobs_dir = OUT_DIR / 'jobs'
jobs_dir.mkdir(exist_ok=True)
for jid in JOB_RESULTS:
    src = EVAL_DIR / 'jobs' / jid
    for fname in ('job.json', 'report.md', 'predict.log', 'evaluate.log'):
        if (src / fname).exists():
            (jobs_dir / jid).mkdir(exist_ok=True)
            shutil.copy2(src / fname, jobs_dir / jid / fname)

readme = [
    '# Text2Cypher final evaluation: ' + EVAL_TAG,
    '',
    'Generated ' + created + ' on Kaggle (session ' + SESSION + ', commit ' + GIT_COMMIT[:10] + ', ' + GPU_LABEL + ').',
    '',
    '## Files',
    '- `tables/headline.csv` (+ `headline.tex`): one row per job, formatted with confidence intervals.',
    '- `tables/overall_metrics.csv`: one row per job, all metrics, CI bounds, latency and provenance (platform, GPU).',
    '- `tables/metrics_by_complexity.csv`, `metrics_by_data_source.csv`, `metrics_by_database.csv`: breakdowns.',
    '- `tables/metrics_by_database_access.csv`, `metrics_by_complexity_and_database_access.csv`: rows with and without a demo database.',
    '- `tables/gold_query_health.csv`: databases where some or all gold queries fail to execute.',
    '- `tables/error_categories.csv`: row counts per outcome category per job.',
    '- `tables/paired_tests.csv`: paired comparisons on identical rows (raw and Holm-adjusted p-values, CIs, TOST for schema families).',
    '- `tables/schema_filtering.csv`: enhanced vs exact-match schema, token efficiency and accuracy (all rows and pruned rows only).',
    '- `tables/training_summary.csv`, `training_curves.csv`: training telemetry merged across sessions.',
    '- `tables/integrity_checks.csv`: sample consistency, adapter verification, commit/platform consistency, gold-query health.',
    '- `tables/jobs.csv`: job registry; `tables/published_baselines.csv`: hand-entered literature numbers.',
    '- `predictions_long.parquet` / `predictions_long.csv.gz`: one row per job and instance.',
    '- `results.json`: all tables (except curves) plus the manifest; `manifest.json`: provenance.',
    '- `figures/`: PNG (300 dpi) and PDF figures; `jobs/`: per-job logs, reports and metadata.',
    '',
    '## Metric definitions',
    '- `google_bleu_corpus`: corpus Google-BLEU over 1-4 grams of Cypher-aware tokens; CI by percentile bootstrap over rows.',
    '- `exact_match`: string equality after trimming and dropping a trailing semicolon. `normalised_exact_match` also ignores whitespace and keyword case. Wilson CIs.',
    '- `execution_accuracy`: share of rows (with a demo database and an executable gold query) whose predicted result set equals the gold result set. Row order is compared only when the gold query has ORDER BY. Wilson CI.',
    '- `execution_accuracy_toolkit`: same numerator over every executed row including failing gold queries (matches `app.py evaluate` reports).',
    '- `executable_rate`: share of executed rows whose prediction ran without error; `executable_rate_gold_ok` restricts it to rows whose gold query runs. `syntax_valid_rate`: share passing EXPLAIN.',
    '- Error categories: correct, wrong_result (ran, different result), invalid_query (fails EXPLAIN: syntax or schema), runtime_error (passes EXPLAIN, fails at run time), execution_error (syntax not checked), empty_output, gold_failed, excluded_dead_db, connection_failed (Neo4j unreachable; not scored), not_executed (no database).',
    '- Paired tests: exact McNemar for binary metrics; paired bootstrap for mean sentence GLEU; Holm correction within each family x metric.',
    '',
    '## Caveats',
    '- Every GPU job uses the same seeded sample of the ' + SPLIT + ' split (n=' + LIMIT_LABEL + ', seed ' + str(SAMPLE_SEED) + '). GGUF CPU jobs use a smaller subset of it.',
    '- s/item is not comparable across platforms or batch sizes (Gemma/Granite use batch 4, Qwen 8). Compare latency only within one platform (see the `platform` column).',
    '- Prompts are fitted to max_seq_length minus the generation reserve by trimming schema property lines, as in training; `Trimmed` counts affected rows per job.',
    '- `Executable` in the headline is computed over rows whose gold query runs (`executable_rate_gold_ok`).',
    '- Databases where every gold query fails (' + (', '.join(DEAD_DBS) or 'none') + ') cannot contribute to execution accuracy; see `gold_query_health.csv`.',
    '- Execution accuracy depends on the Neo4j Labs demo server; rows without a database are excluded from it.',
    '- Gemma 4 E2B is a base (pre-trained) checkpoint without a chat template, so its zero-shot prompt uses the plain-text fallback (labelled in the Condition column).',
    '- Bootstrap p-values below 1/BOOTSTRAP_SAMPLES are reported as < ' + format(1 / BOOTSTRAP_SAMPLES, '.2g') + ' and floored at that value before Holm adjustment, so their Holm p is an upper bound.',
    '- Schema-filtering equivalence: TOST at margin +/-' + str(EQUIV_MARGIN) + ' via the (1 - 2 alpha) paired bootstrap CI.',
    '- `cross_schema` jobs score the enhanced-trained adapter with exact-match prompts (inference-time effect of schema filtering).',
    '- Percentile bootstrap intervals for corpus GLEU are reported as computed and need not contain the point estimate; the figures clip error bars at the estimate.',
    '- `logged_training_hours` is the span of logged steps per session; it excludes model loading, tokenisation and uploads.',
    '- `trainer_train_loss_final_session` is reported by the Trainer for the final session only.',
    '- Published baselines come from other papers with different samples and protocols; compare them descriptively only.',
]
(OUT_DIR / 'README.md').write_text(LF.join(readme) + LF, encoding='utf-8')

if MANIFEST['mixed_commits'] or MANIFEST['mixed_platforms']:
    print('WARNING: this bundle mixes commits ' + str([c[:10] for c in BUNDLE_COMMITS]) + ' and/or platforms '
          + str(BUNDLE_PLATFORMS) + '. It is not a single-protocol result; follow the re-run plan.')

# Bundle and zip go to /kaggle/working so they are saved with the notebook version.
SAVED_DIR = Path(OUTPUT_DIR) / 'thesis-results' / EVAL_TAG / SESSION
shutil.copytree(OUT_DIR, SAVED_DIR, dirs_exist_ok=True)
archive = shutil.make_archive(str(Path(OUTPUT_DIR) / ('thesis-results-' + EVAL_TAG + '-' + SESSION)), 'zip',
                              root_dir=OUT_DIR)
uploaded = hub_upload_folder(OUT_DIR, HUB_EVAL_PREFIX + '/thesis/' + SESSION,
                             '[final-eval] thesis bundle ' + EVAL_TAG + ' ' + SESSION)
print('Results bundle : ' + str(SAVED_DIR) + ' (saved with the version)')
print('Zip archive    : ' + archive)
print('Logs           : ' + str(LOG_DIR))
if UPLOAD_TO_HUB:
    print('Hub upload     : ' + ('OK' if uploaded else 'FAILED') + ' (https://huggingface.co/' + HUB_REPO
          + '/tree/main/' + HUB_EVAL_PREFIX + '/thesis/' + SESSION + ')')

summary = ['Session ' + SESSION + ' finished (' + EVAL_TAG + ').']
for _, r in OVERALL.iterrows():
    summary.append(str(r['job_id']) + ': EA ' + _fmt(r.get('execution_accuracy')) + ', GLEU '
                   + _fmt(r.get('google_bleu_corpus')) + ', EM ' + _fmt(r.get('exact_match')))
if MANIFEST['jobs_not_run']:
    summary.append('Not run: ' + ', '.join(MANIFEST['jobs_not_run']) + '. Commit a new version to finish them.')
if len(failed_checks):
    summary.append(str(len(failed_checks)) + ' integrity check(s) failed; see integrity_checks.csv.')
STATUS['outcome'] = 'completed'
notify('finished', LF.join(summary),
       tags=['warning'] if len(failed_checks) or MANIFEST['jobs_not_run'] else ['white_check_mark'])
display(HEADLINE)